# 第五章 动手搭建大模型 · 代码汇总

> 原文：[第五章 动手搭建大模型.md](./第五章%20动手搭建大模型.md)  
> 参考代码：[`code/`](./code)

本 Notebook 按原文顺序汇总第 5 章的 Python 源码，保留原有中文注释，并对 **GQA、RoPE、注意力维度、损失掩码、训练流程** 等关键部分补充了注释与说明。

**运行说明**

- 5.1 模型部分可直接运行，依赖 `torch`、`transformers`。
- 5.2 Tokenizer 部分需要准备 JSONL 语料后才能训练。
- 5.3 数据下载、预训练与 SFT 涉及大数据和 GPU，相关入口默认关闭：
  - 预训练 / SFT：`RUN_TRAINING = False`
  - 文本生成：`RUN_SAMPLING = False`
  - 需要运行时改为 `True`，并准备好数据与权重路径。

## 0. 环境与导入

```bash
pip install torch transformers tokenizers datasets tqdm pandas swanlab
```

`swanlab` 仅用于训练日志记录；未安装时下面的代码仍可正常阅读和定义。

In [2]:
import math
from typing import Optional, Tuple

import torch
import torch.nn.functional as F
from torch import nn

from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

# 固定随机种子，保证示例可复现
torch.manual_seed(42)

## 5.1 动手实现一个 LLaMA2 大模型

### 5.1.1 定义超参数

关键点：

- `dim` 是残差流宽度，Attention、MLP 的输出最终都回到 `dim`。
- `n_kv_heads < n_heads` 即 **GQA（分组查询注意力）**：K/V 头更少，减少 KV Cache 与计算量。
- `hidden_dim=None` 时，FFN 中间维度按 `4 * dim * 2 / 3` 计算，并向上取整到 `multiple_of` 的倍数。
- 原文后续示例中的 `args` 都指下面实例化的 `ModelConfig()`。

In [3]:
class ModelConfig(PretrainedConfig):
    """LLaMA2 风格小模型的超参数配置。"""

    model_type = "Tiny-K"

    def __init__(
            self,
            dim: int = 768,              # 模型隐藏维度（残差流宽度）
            n_layers: int = 12,          # Decoder 层数
            n_heads: int = 16,           # 查询头数
            n_kv_heads: int = 8,         # K/V 头数；小于 n_heads 即为 GQA
            vocab_size: int = 6144,      # 词表大小
            hidden_dim: int = None,      # FFN 中间维度；None 时按公式自动计算
            multiple_of: int = 64,       # FFN 中间维度需为它的整数倍
            norm_eps: float = 1e-5,      # RMSNorm 的数值稳定项
            max_seq_len: int = 512,      # 最大上下文长度
            dropout: float = 0.0,        # Dropout 概率
            flash_attn: bool = True,     # 是否希望使用 Flash Attention
            **kwargs,
    ):
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.multiple_of = multiple_of
        self.norm_eps = norm_eps
        self.max_seq_len = max_seq_len
        self.dropout = dropout
        self.flash_attn = flash_attn
        super().__init__(**kwargs)

In [4]:
# 原文示例中直接使用了 args，这里显式实例化，便于后续单元格运行
args = ModelConfig()

print(f"dim={args.dim}, n_layers={args.n_layers}, n_heads={args.n_heads}, n_kv_heads={args.n_kv_heads}")
print(f"head_dim={args.dim // args.n_heads}, FFN hidden_dim={args.hidden_dim or '自动计算'}")
print(f"GQA 复制倍数 n_rep={args.n_heads // args.n_kv_heads}")

dim=768, n_layers=12, n_heads=16, n_kv_heads=8
head_dim=48, FFN hidden_dim=自动计算
GQA 复制倍数 n_rep=2


### 5.1.2 构建 RMSNorm

与 LayerNorm 的区别：**不减均值，只除以均方根**，因此少了一次均值计算，参数也只有缩放项 `γ`。

$$
\text{RMSNorm}(x)=\frac{x}{\sqrt{\frac{1}{n}\sum_{i=1}^n x_i^2+\epsilon}}\cdot\gamma
$$

代码中先转为 `float32` 计算，避免半精度下平方溢出，最后再恢复原 dtype。

In [5]:
class RMSNorm(nn.Module):
    """只做均方根缩放、不减均值的归一化。"""

    def __init__(self, dim: int, eps: float):
        super().__init__()
        # eps 防止输入全为 0 时除以 0
        self.eps = eps
        # 可学习的缩放参数，初始化为 1
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        # RMS(x) = sqrt(mean(x^2) + eps)
        # rsqrt 即 1 / sqrt(...)，得到每个位置、每个 token 的缩放因子
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        # 先转 float32 计算，避免半精度下平方溢出，再恢复原 dtype
        output = self._norm(x.float()).type_as(x)
        # 逐元素乘以可学习权重 γ
        return output * self.weight

In [6]:
norm = RMSNorm(args.dim, args.norm_eps)
x = torch.randn(1, 50, args.dim)
output = norm(x)
print(output.shape)

torch.Size([1, 50, 768])


### 5.1.3 构建 LLaMA2 Attention

LLaMA2 的注意力有两个关键设计：

1. **GQA**：Q 有 `n_heads` 个头，K/V 只有 `n_kv_heads` 个头。计算前用 `repeat_kv` 把每个 K/V 头复制 `n_rep = n_heads / n_kv_heads` 份，使头数与 Q 对齐。
2. **RoPE**：对 Q、K 的相邻维度对做二维旋转，把位置信息编码为相对位置；V 不做旋转。

#### 5.1.3.1 repeat_kv

In [7]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """把 n_kv_heads 个 K/V 头复制 n_rep 份，使其数量与 Q 头一致。

    输入 x:  (batch, seq_len, n_kv_heads, head_dim)
    输出:    (batch, seq_len, n_kv_heads * n_rep, head_dim)
    """
    bs, slen, n_kv_heads, head_dim = x.shape

    # 若 Q 与 K/V 头数相同，无需复制
    if n_rep == 1:
        return x

    # (B,T,H,D) -> (B,T,H,1,D) -> (B,T,H,n_rep,D) -> (B,T,H*n_rep,D)
    return (
        x[:, :, :, None, :]
        .expand(bs, slen, n_kv_heads, n_rep, head_dim)
        .reshape(bs, slen, n_kv_heads * n_rep, head_dim)
    )

In [8]:
# 小例子：4 个 K/V 头复制 2 份 -> 8 个头
x = torch.randn(1, 10, 4, 96)
print("repeat_kv:", tuple(x.shape), "->", tuple(repeat_kv(x, 2).shape))

repeat_kv: (1, 10, 4, 96) -> (1, 10, 8, 96)


##### 为什么 GQA 选择“复制 K/V 头”而不是“加投影层”？
这是 LLaMA2 70B 中 GQA（Grouped-Query Attention）的核心设计问题，答案主要涉及**显存带宽、推理效率和设计意图**三点。
###### 1. GQA 的初衷就是为了省 KV Cache，复制不增加显存
GQA 出现的背景是 MHA（多头注意力）中 KV Cache 太大：
```
KV Cache 大小 = 2 × batch × seq_len × n_heads × head_dim
```
推理时这是主要显存瓶颈。GQA 让 K/V 只存 `n_kv_heads` 份（如 8 个），而 Q 保持 `n_heads`（如 64 个），**KV Cache 直接缩小 n_rep 倍**。
关键在于：这个节省发生在 **KV Cache 的存储层面**（显存中的 persistent tensor 只有 n_kv_heads 份）。而 `repeat_kv` 只是在**每一步前向计算时临时展开**，生成的是视图（expand）+ reshaped 后的临时张量，用完即弃，**不参与 Cache 持久化**。
如果反过来给 K/V 加投影层升到 n_heads，那 Cache 就得存 n_heads 份——GQA 就白做了。
###### 2. 如果用隐藏层转换，等价于 MHA，失去 GQA 意义
你说的“加一层投影”（比如 `W_k: d_model → n_heads × head_dim` 把 8 个 KV 头映射成 64 个）在数学上就是 MHA。这样：
- KV Cache 回到原始大小，显存优势消失
- 每步预填充/解码的计算量和带宽回到 MHA 水平
而 GQA 的设计哲学是：**多个 Q 头共享同一组 K/V 头**（组内共享），实验证明这带来的质量损失极小（论文中 GQA-8 几乎匹配 MHA），但换取巨大的推理加速。
###### 3. 复制是“零成本”的，投影是“真成本”的
```python
x[:, :, :, None, :].expand(...)  # expand 是视图操作，不复制数据
.reshape(...)                    # reshape 因 expand 的 stride 不连续，可能触发一次实际拷贝
```
- **expand** 只是改 stride，不写内存，O(1)
- 后续 `reshape` + matmul 时，硬件层面本来就是**广播读取**：K/V 的同一块数据被多个 Q 头的矩阵乘法重复读取，天然契合
而加一个线性层意味着：
- 额外的权重参数和矩阵乘 FLOPs
- 每 token 都要执行，增加延迟
- 破坏“共享”结构——投影出的 64 个 K 头各不相同，等于回到 MHA
###### 总结对比
| 方案 | KV Cache | 额外参数 | 额外计算 | 效果 |
|---|---|---|---|---|
| repeat_kv（GQA）| n_kv_heads 份 ✓ | 0 | 接近 0（广播）| 质量略降，速度大增 |
| 加投影层（=MHA）| n_heads 份 ✗ | 有 | 有 | 无任何收益 |
**一句话**：`repeat_kv` 不是“补齐维度”的功能性操作，而是 GQA “少存 K/V、多组 Q 共享”这一设计在计算图上的实现方式——复制是逻辑上的视图展开，物理上 K/V 始终只有 n_kv_heads 份，这正是省显存、提速的关键。

#### 5.1.3.2 旋转位置编码（RoPE）

- `precompute_freqs_cis(dim, end)` 中的 `dim` 是 **head_dim**，返回 `(seq_len, head_dim/2)` 的 cos/sin。
- `reshape_for_broadcast` 把频率整理成 `(1, T, 1, D/2)`，以便与 `(B, T, H, D/2)` 广播。
- `apply_rotary_emb` 把最后一维拆成 `(D/2, 2)`，对每对 `(x_r, x_i)` 应用旋转矩阵。

##### reshape_for_broadcast 条件：`i == 1 or i == ndim - 1`
针对 `x` 的四维形状 `(B, T, H, D)`，只有两个维度需要保留频率值：
| 维度索引 | 含义 | 是否保留 | 原因 |
|---|---|---|---|
| i = 0 | batch | ❌ 置 1 | 频率与 batch 无关，所有样本共用 |
| **i = 1** | seq_len | ✅ 保留 T | **每个位置有自己的旋转角** |
| i = 2 | heads | ❌ 置 1 | 所有的 Q/K 头共用同一组旋转角 |
| **i = ndim-1 = 3** | head_dim | ✅ 保留 D/2 | **每个维度对有自己不同的频率** |


##### apply_rotary_emb解释：“实部/虚部”只是我们强加的**约定**，不是数据自带的属性
这是一个非常好的问题，触及了 RoPE 实现的本质。
###### 数据里本来就没有复数
`xq` 里存的全是普通的实数（投影后的 Q 向量分量）。所谓“实部、虚部”**不是从数据里检测出来的**，而是我们人为规定：
> 把两个相邻的实数 `(x_{2i}, x_{2i+1})` **看作**复平面上的一个点：
> - 偶数位 `x_{2i}` → 充当“实部”（x 坐标）
> - 奇数位 `x_{2i+1}` → 充当“虚部”（y 坐标）
即虚构一个复数 $z_i = x_{2i} + x_{2i+1} \cdot j$。数据不变，只是**解释方式**变了。
###### 类比：坐标轴也是人为命名的
一个二维平面上的点 `(3, 5)`：
- 你可以叫它“x 坐标 = 3，y 坐标 = 5”
- 也可以叫它“实部 = 3，虚部 = 5”
- 点还是那个点，只是给两个分量起了不同的名字
`unbind(-1)` 拆的只是“每对的第 0 个元素”和“第 1 个元素”——两个普通的实数子张量。叫它们 `xq_r` / `xq_i` 只是变量命名上的比喻。
###### 为什么强行这样“解释”是合法且有用的？
因为复数乘法在数学上**恰好等价于**二维向量的旋转。把两个实数当复数乘以 $e^{i\theta}$：
$$(x_r + x_i j)(\cos\theta + i\sin\theta)$$
展开后得到的公式：
```python
xq_out_r = xq_r * cos - xq_i * sin
xq_out_i = xq_r * sin + xq_i * cos
```
这正是标准的二维旋转矩阵作用：
$$\begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} x_r \\ x_i \end{pmatrix}$$
也就是说：**复数只是二维旋转的便捷记号**。你完全可以不用“复数”这个词，直接说“对每个相邻维度对做平面旋转”，代码一模一样。复数框架的价值在于：
1. 数学推导简洁（$q \cdot k$ 内积只依赖相对位置 $m-n$ 的证明用复数指数性质一目了然）
2. 旋转操作有现成的代数表达
###### 所以整个流程是
```
实数向量 (D 维)
   ↓ reshape: 两两配对           ← 人为分组
D/2 个二维点
   ↓ unbind: 拆成两个分量         ← 人为命名为 r / i
两个实数张量
   ↓ 乘 cos/sin 组合              ← 本质是二维旋转，借复数记号推导
旋转后的 D/2 个二维点
   ↓ stack + flatten
旋转后的实数向量 (D 维)
```
**全程没有复数类型参与**（PyTorch 有 `torch.complex`，但这份实现没用它，纯实数运算）。数据的“复数性”只存在于我们的脑中和推导的草稿纸上。
###### 一句话总结
实数当然能“分出实部和虚部”——因为不是在分离已有的复数，而是在**配对 + 命名**：把每两个相邻实数约定为复数的两个分量，从而套用复数乘法 = 二维旋转的数学工具。这也正是 RoPE 全称 “Rotary **Position Embedding**” 的含义：旋转的是嵌入向量，而非什么复数信号。

In [11]:
# 注意：此处的 dim 应为 head_dim = dim // n_head，因为 RoPE 是对每个头分别旋转
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    """预计算每个位置、每个维度对的 cos/sin。

    返回:
        freqs_cos, freqs_sin: 形状均为 (end, dim // 2)
    """
    # 频率：1 / (theta ^ (2i / dim))，i = 0, 1, ..., dim/2 - 1
    # torch.arange(0, dim, 2) 生成偶数序列 [0, 2, 4, ..., dim-2]，共 dim/2 个元素
    # [: (dim // 2)] 截取前 dim//2 个
    # ... / dim 得到 2i / dim
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    # 位置序列 0, 1, ..., end-1
    t = torch.arange(end, device=freqs.device)
    # 外积得到每个位置的旋转角度，形状 (end, dim // 2)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cos(freqs)
    freqs_sin = torch.sin(freqs)
    return freqs_cos, freqs_sin


def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor):
    """把 (T, D/2) 的频率张量整理成可与 (B,T,H,D/2) 广播的形状。"""
    ndim = x.ndim
    assert 0 <= 1 < ndim
    assert freqs_cis.shape == (x.shape[1], x.shape[-1])

    # 只保留序列维和最后一维，其余维度置 1
    # enumerate(x.shape) 同时给出每个维度的索引 i 和大小 d
    # d if 条件 else 1    条件成立 → 保留原大小 d；否则 → 置为 1
    shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
    return freqs_cis.view(shape)


def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cos: torch.Tensor,
    freqs_sin: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    """对 Q、K 按相邻维度对 (x_{2i}, x_{2i+1}) 施加二维旋转。

    输入/输出:
        xq, xk: (B, T, n_heads, head_dim)
    """
    # 把最后一维拆成 (D/2, 2)，得到每对元素的“实部”和“虚部”
    # shape[:-1] + (-1, 2)构造目标形状(xq.shape[:-1] 是除最后一维外的所有维度，+ 是元组拼接)。
    # xq: (B, T, H, D) 如 xq=(2, 128, 32, 64); xq.shape[:-1] = (2, 128, 32); 
    # 拼接 (-1, 2) → (2, 128, 32, -1, 2) ，-1：让 PyTorch 自动算出 = D/2（维度对个数）2：每对固定两个元素
    # .unbind(-1) 沿最后一维（大小为 2 的那维）拆开，返回两个张量组成的元组。
    xq_r, xq_i = xq.float().reshape(xq.shape[:-1] + (-1, 2)).unbind(-1)
    xk_r, xk_i = xk.float().reshape(xk.shape[:-1] + (-1, 2)).unbind(-1)

    # 频率广播到 (1, T, 1, D/2)
    freqs_cos = reshape_for_broadcast(freqs_cos, xq_r)
    freqs_sin = reshape_for_broadcast(freqs_sin, xq_r)

    # 二维旋转矩阵: [cos -sin; sin cos] @ [r; i]
    xq_out_r = xq_r * freqs_cos - xq_i * freqs_sin
    xq_out_i = xq_r * freqs_sin + xq_i * freqs_cos
    xk_out_r = xk_r * freqs_cos - xk_i * freqs_sin
    xk_out_i = xk_r * freqs_sin + xk_i * freqs_cos

    # 把实部、虚部重新交错拼回最后一维
    xq_out = torch.stack([xq_out_r, xq_out_i], dim=-1).flatten(3)
    xk_out = torch.stack([xk_out_r, xk_out_i], dim=-1).flatten(3)

    return xq_out.type_as(xq), xk_out.type_as(xk)

In [10]:
xq = torch.randn(1, 50, 6, 48) # bs, seq_len, dim//n_head, n_head_dim
xk = torch.randn(1, 50, 6, 48) # bs, seq_len, dim//n_head, n_head_dim

# 使用 precompute_freqs_cis 函数获取 sin和cos
cos, sin = precompute_freqs_cis(288//6, 50)
print(cos.shape, sin.shape)
xq_out, xk_out = apply_rotary_emb(xq, xk, cos, sin)

xq_out.shape, xk_out.shape

torch.Size([50, 24]) torch.Size([50, 24])


(torch.Size([1, 50, 6, 48]), torch.Size([1, 50, 6, 48]))

#### 5.1.3.3 组装 Attention

维度流转（以 `args.dim=768, n_heads=16, n_kv_heads=8` 为例）：

| 步骤 | 形状 |
| --- | --- |
| 输入 `x` | `(B, T, 768)` |
| `wq(x)` | `(B, T, 16 × 48)` |
| `wk(x)` / `wv(x)` | `(B, T, 8 × 48)` |
| 拆头后 Q | `(B, T, 16, 48)` |
| 拆头后 K/V | `(B, T, 8, 48)` |
| `repeat_kv` 后 K/V | `(B, T, 16, 48)` |
| transpose 后 | `(B, 16, T, 48)` |
| 注意力输出合并 | `(B, T, 768)` |

`scaled_dot_product_attention(..., is_causal=True)` 会在内部生成因果掩码，因此无需显式构造上三角矩阵。

In [12]:
class Attention(nn.Module):
    def __init__(self, args: ModelConfig):
        super().__init__()
        # 未指定 n_kv_heads 时退化为普通多头注意力
        self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        assert args.n_heads % self.n_kv_heads == 0

        # 这里不做模型并行，因此 local 头数等于总头数
        model_parallel_size = 1
        self.n_local_heads = args.n_heads // model_parallel_size
        self.n_local_kv_heads = self.n_kv_heads // model_parallel_size
        # 每个 K/V 头需要被复制的次数，使 K/V 头数与 Q 头数对齐
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        # 每个头的维度
        self.head_dim = args.dim // args.n_heads

        # Q 投影输出 n_heads * head_dim；K/V 只输出 n_kv_heads * head_dim
        self.wq = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(args.n_heads * self.head_dim, args.dim, bias=False)

        self.attn_dropout = nn.Dropout(args.dropout)
        self.resid_dropout = nn.Dropout(args.dropout)
        self.dropout = args.dropout

        # PyTorch >= 2.0 提供 scaled_dot_product_attention，可直接走 Flash Attention
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # 上三角为 -inf 的因果掩码，保证位置 t 只能看到 <= t 的位置
            mask = torch.full((1, 1, args.max_seq_len, args.max_seq_len), float("-inf"))
            mask = torch.triu(mask, diagonal=1)
            self.register_buffer("mask", mask)

    def forward(self, x: torch.Tensor, freqs_cos: torch.Tensor, freqs_sin: torch.Tensor):
        # x: (B, T, dim)
        bsz, seqlen, _ = x.shape

        # 线性投影: Q 为 (B,T,n_heads,head_dim)，K/V 为 (B,T,n_kv_heads,head_dim)
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)
        xq = xq.view(bsz, seqlen, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seqlen, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seqlen, self.n_local_kv_heads, self.head_dim)

        # 只对 Q、K 施加 RoPE；V 不需要位置旋转
        xq, xk = apply_rotary_emb(xq, xk, freqs_cos, freqs_sin)

        # GQA：把 K/V 头复制到与 Q 头数相同
        xk = repeat_kv(xk, self.n_rep)
        xv = repeat_kv(xv, self.n_rep)

        # 把头维提到 batch 维: (B, n_heads, T, head_dim)
        xq = xq.transpose(1, 2)
        xk = xk.transpose(1, 2)
        xv = xv.transpose(1, 2)

        if self.flash:
            # is_causal=True 内部生成因果掩码，无需显式 mask
            output = torch.nn.functional.scaled_dot_product_attention(
                xq, xk, xv,
                attn_mask=None,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=True,
            )
        else:
            # 手动实现: scores = QK^T / sqrt(d)，加因果掩码后 softmax
            scores = torch.matmul(xq, xk.transpose(2, 3)) / math.sqrt(self.head_dim)
            assert hasattr(self, 'mask')
            scores = scores + self.mask[:, :, :seqlen, :seqlen]
            scores = F.softmax(scores.float(), dim=-1).type_as(xq)
            scores = self.attn_dropout(scores)
            output = torch.matmul(scores, xv)

        # (B, n_heads, T, head_dim) -> (B, T, n_heads * head_dim)
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)

        # 输出投影回残差流维度 dim
        output = self.wo(output)
        output = self.resid_dropout(output)
        return output

In [13]:
# 创建Attention实例
attention_model = Attention(args)

# 模拟输入数据
batch_size = 1
seq_len = 50  # 假设实际使用的序列长度为50
dim = args.dim
x = torch.rand(batch_size, seq_len, dim)  # 随机生成输入张量
# freqs_cos = torch.rand(seq_len, dim // 2)  # 模拟cos频率，用于RoPE
# freqs_sin = torch.rand(seq_len, dim // 2)  # 模拟sin频率，用于RoPE

freqs_cos, freqs_sin = precompute_freqs_cis(dim//args.n_heads, seq_len)

# 运行Attention模型
output = attention_model(x, freqs_cos, freqs_sin)

# attention出来之后的形状 依然是[batch_size, seq_len, dim]
print("Output shape:", output.shape)

Output shape: torch.Size([1, 50, 768])


### 5.1.4 构建 LLaMA2 MLP

LLaMA 使用 **SwiGLU**，包含三个线性层：

$$
\text{MLP}(x)=W_2\big(\text{SiLU}(W_1x)\odot W_3x\big)
$$

- `w1`、`w3` 把维度从 `dim` 升到 `hidden_dim`；
- 两者逐元素相乘（门控）；
- `w2` 再降回 `dim`。

默认 `hidden_dim` 约为 `8/3 * dim`，并向上取整到 `multiple_of` 的倍数。

In [14]:
class MLP(nn.Module):
    """LLaMA 使用的 SwiGLU 前馈网络。"""

    def __init__(self, dim: int, hidden_dim: int, multiple_of: int, dropout: float):
        super().__init__()
        # 默认中间维度 = 4 * dim * 2/3，并向上取整到 multiple_of 的倍数
        if hidden_dim is None:
            hidden_dim = 4 * dim
            hidden_dim = int(2 * hidden_dim / 3)
            hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of)

        self.w1 = nn.Linear(dim, hidden_dim, bias=False)       # 门控分支
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)       # 降维回 dim
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)       # 值分支
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # SwiGLU: w2( SiLU(w1(x)) * w3(x) )
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))

In [15]:
# 创建MLP实例
mlp = MLP(args.dim, args.hidden_dim, args.multiple_of, args.dropout)
# 随机生成数据
x = torch.randn(1, 50, args.dim)
# 运行MLP模型
output = mlp(x)
print(output.shape)

torch.Size([1, 50, 768])


### 5.1.5 LLaMA2 Decoder Layer

每个 Decoder Layer 由两个 **Pre-Norm 子层** 组成：

1. `x = x + Attention(RMSNorm(x))`
2. `x = x + MLP(RMSNorm(x))`

先归一化再进子层，可以让残差主路保持恒等映射，深层训练更稳定。输入输出形状始终是 `(B, T, dim)`。

##### 为什么要定义两个“看起来一样”的 RMSNorm？
这两行的确创建的是**结构完全相同**的模块（同样的 `dim` 和 `eps`），但它们是**两个独立的对象，拥有各自独立的可学习参数**。这一点是关键。
###### 1. RMSNorm 不是无参数的操作，它含有可学习的权重
RMSNorm 的公式：
$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}} \cdot \gamma$$
其中 $\gamma$（`weight`）是一个**可学习的缩放向量**，形状 `(dim,)`，随训练更新。
```python
self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)  # 有自己的 γ₁
self.ffn_norm      = RMSNorm(args.dim, eps=args.norm_eps)  # 有自己的 γ₂
```
如果把两行合并成一个实例共用，那么两个子层就共享同一个 γ，**模型容量下降，且梯度信号混在一起**。
###### 2. 两个子层需要的“归一化后处理”不同
注意力子层和前馈子层的功能、激活分布完全不同：
- **attention_norm** 的输出喂给 Q/K/V 投影 → γ 学到的缩放服务于“让注意力打分更稳定”
- **ffn_norm** 的输出喂给 W1/W3 的 MLP → γ 学到的缩放服务于“让激活函数工作在合适区间”
两者的输入分布、下游需求不同，训练后会收敛到**不同的 γ 值**。实验和经验都表明：让每个位置拥有独立的 scale 参数效果更好，而共享会带来性能损失。
###### 3. 这是 Transformer 的标准结构：每个子层各配一个 Norm
Transformer 的每个子层（attention / FFN）都有属于自己的 norm：
```python
h = x + Attention(Norm₁(x))     # 子层 1 专属 Norm₁
out = h + FFN(Norm₂(h))         # 子层 2 专属 Norm₂
```
回看 LLaMA 源码也是一样的写法。可以数一下：32 层 × 每层 2 个 RMSNorm = 64 个 norm 模块，每个都有独立参数。
###### 4. 打个比方
就像两间房的空调都是同一型号（同样的 `dim`、`eps`），但各自有**独立的遥控器和设定温度**：
- “型号相同” = 结构参数一样
- “设定不同” = 可学习权重不同
写 `self.ffn_norm = self.attention_norm`（共享实例）虽然语法合法、代码更短，但等于两间房共用一个温控——不符合设计意图。
###### 总结
| 视角 | 两个 norm 的关系 |
|---|---|
| 结构/超参 | 完全相同（都是 `RMSNorm(dim, eps)`） |
| 实例/参数 | **完全独立**，各自维护一个 γ 向量 |
| 功能 | 分别适配注意力子层和 FFN 子层的不同需求 |
所以“定义两次相同初始化”不是冗余，而是因为 **RMSNorm 带可学习参数，且每个子层需要学出属于自己的归一化缩放**。若真的共享一个实例，模型仍然能跑，但等于人为引入了参数约束，通常会造成性能退化。

In [16]:
class DecoderLayer(nn.Module):
    def __init__(self, layer_id: int, args: ModelConfig):
        super().__init__()
        self.n_heads = args.n_heads
        self.dim = args.dim
        self.head_dim = args.dim // args.n_heads
        self.attention = Attention(args)
        self.feed_forward = MLP(
            dim=args.dim,
            hidden_dim=args.hidden_dim,
            multiple_of=args.multiple_of,
            dropout=args.dropout,
        )
        self.layer_id = layer_id
        # Pre-Norm：先归一化再进子层
        self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps)

    def forward(self, x, freqs_cos, freqs_sin):
        # 注意力子层 + 残差
        h = x + self.attention.forward(self.attention_norm(x), freqs_cos, freqs_sin)
        # 前馈子层 + 残差
        out = h + self.feed_forward.forward(self.ffn_norm(h))
        return out

In [17]:
# 创建LLaMADecoderLayer实例
decoderlayer = DecoderLayer(0, args)

# 模拟输入数据
dim = args.dim
seq_len = 50

x = torch.randn(1, seq_len, dim) # [bs, seq_len, dim]

freqs_cos, freqs_sin = precompute_freqs_cis(dim//args.n_heads, seq_len)

out = decoderlayer(x, freqs_cos, freqs_sin)

print(out.shape) # 形状和输入的x一样 [batch_size, seq_len, dim]

torch.Size([1, 50, 768])


### 5.1.6 构建 LLaMA2 模型

关键点：

- **权重共享**：`tok_embeddings.weight` 与 `output.weight` 指向同一矩阵，减少词表相关的参数量。
- **RoPE 频率**：只与位置有关，注册为 `buffer`，不参与训练。
- **损失**：`reduction='none'` 返回逐 token 损失，上层再用 `loss_mask` 过滤 padding；`ignore_index=0` 会忽略 id 为 0 的 token。
- **推理优化**：`targets=None` 时只计算最后一个位置的 logits，避免生成时对整段序列做输出投影。
- **`generate`**：教学版逐 token 重新前向，没有使用 KV Cache；`temperature=0` 为贪婪解码。

In [18]:
class Transformer(PreTrainedModel):
    config_class = ModelConfig
    last_loss: Optional[torch.Tensor]

    def __init__(self, args: ModelConfig = None):
        super().__init__(args)
        self.args = args
        self.vocab_size = args.vocab_size
        self.n_layers = args.n_layers

        # token Embedding: (B,T) -> (B,T,dim)
        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)
        self.dropout = nn.Dropout(args.dropout)

        self.layers = torch.nn.ModuleList()
        for layer_id in range(args.n_layers):
            self.layers.append(DecoderLayer(layer_id, args))

        # 所有层之后做一次 RMSNorm
        self.norm = RMSNorm(args.dim, eps=args.norm_eps)
        # 输出投影到词表维度
        self.output = nn.Linear(args.dim, args.vocab_size, bias=False)

        # 权重共享：输入 Embedding 与输出投影使用同一矩阵，减少参数
        self.tok_embeddings.weight = self.output.weight

        # RoPE 频率只与位置有关，预先算好并注册为 buffer（不参与训练）
        freqs_cos, freqs_sin = precompute_freqs_cis(
            self.args.dim // self.args.n_heads, self.args.max_seq_len
        )
        self.register_buffer("freqs_cos", freqs_cos, persistent=False)
        self.register_buffer("freqs_sin", freqs_sin, persistent=False)

        self.apply(self._init_weights)
        # 对残差分支的投影做更小的初始化，缓解深层残差累积
        for pn, p in self.named_parameters():
            if pn.endswith('w3.weight') or pn.endswith('wo.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * args.n_layers))

        self.last_loss = None
        # 复用的输出容器，避免每次 forward 新建对象
        self.OUT = CausalLMOutputWithPast()
        self._no_split_modules = [name for name, _ in self.named_modules()]

    def _init_weights(self, module):
        # 线性层与 Embedding 使用 N(0, 0.02) 初始化，bias 置零
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, tokens: torch.Tensor, targets: Optional[torch.Tensor] = None, **kwargs) -> torch.Tensor:
        """
        - tokens: (B, T) 的 int token 序列
        - targets: (B, T) 的下一个 token，训练时传入
        - 返回 CausalLMOutputWithPast(logits, last_loss)
        """
        # 兼容 HuggingFace 风格的参数名
        if 'input_ids' in kwargs:
            tokens = kwargs['input_ids']
        if 'labels' in kwargs:
            targets = kwargs['labels']

        _bsz, seqlen = tokens.shape
        # (B,T) -> (B,T,dim)
        h = self.tok_embeddings(tokens)
        h = self.dropout(h)

        # 按当前序列长度截取预计算的 RoPE 频率
        freqs_cos = self.freqs_cos[:seqlen]
        freqs_sin = self.freqs_sin[:seqlen]

        for layer in self.layers:
            h = layer(h, freqs_cos, freqs_sin)
        h = self.norm(h)

        if targets is not None:
            # 训练：对每个位置输出 logits，并计算逐 token 交叉熵
            logits = self.output(h)
            # reduction='none' 便于上层用 loss_mask 过滤 padding；0 号 token 被忽略
            self.last_loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=0,
                reduction='none',
            )
        else:
            # 推理：只需要最后一个位置的 logits，减少计算量
            logits = self.output(h[:, [-1], :])
            self.last_loss = None

        self.OUT.__setitem__('logits', logits)
        self.OUT.__setitem__('last_loss', self.last_loss)
        return self.OUT

    @torch.inference_mode()
    def generate(self, idx, stop_id=None, max_new_tokens=256, temperature=1.0, top_k=None):
        """自回归采样；未使用 KV Cache，适合教学演示。

        idx: (B, T) 的 prompt token
        stop_id: 遇到该 token 停止；None 表示不提前停止
        temperature=0 时退化为贪婪解码
        """
        index = idx.shape[1]
        for _ in range(max_new_tokens):
            # 超过最大上下文时，只保留最近的 max_seq_len 个 token
            idx_cond = idx if idx.size(1) <= self.args.max_seq_len else idx[:, -self.args.max_seq_len:]

            # 前向只返回最后一个位置的 logits: (B, 1, vocab)
            logits = self(idx_cond).logits
            logits = logits[:, -1, :]

            if temperature == 0.0:
                # 贪婪解码：取概率最大的 token
                _, idx_next = torch.topk(logits, k=1, dim=-1)
            else:
                # 温度缩放 + top-k 过滤 + 多项式采样
                logits = logits / temperature
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float('Inf')
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)

            # stop_id 为 None 时，Tensor == None 恒为 False，不会提前结束
            if idx_next == stop_id:
                break

            idx = torch.cat((idx, idx_next), dim=1)

        return idx[:, index:]

In [19]:
# LLaMA2Model.forward 接受两个参数，tokens和targets，其中tokens是输入的张量, 应为int类型
x = torch.randint(0, 6144, (1, 50)) # [bs, seq_len]
# 实例化LLaMA2Model
model = Transformer(args=args)
# 计算model的全部参数
num_params = sum(p.numel() for p in model.parameters())
print('Number of parameters:', num_params)

out = model(x)
print(out.logits.shape) # [batch_size, 1, vocab_size]

Number of parameters: 82594560
torch.Size([1, 1, 6144])


#### 上述代码的 LLaMA2 模型结构详解
这份 `Transformer` 类实现的就是一个完整的 LLaMA2 风格 decoder-only 语言模型。整体结构可以用一张图概括：
```
tokens (B,T)
    │
    ▼
tok_embeddings (nn.Embedding)          # 词嵌入
    │
    ▼
dropout
    │
    ▼
┌───────────────────────────┐
│  DecoderLayer × n_layers  │   × N 层堆叠（如 LLaMA2-7B 为 32 层）
│  ┌─────────────────────┐  │
│  │ RMSNorm             │  │  ← Pre-Norm
│  │ Attention (GQA+RoPE)│  │
│  │ + 残差              │  │
│  │ RMSNorm             │  │
│  │ MLP (SwiGLU)        │  │
│  │ + 残差              │  │
│  └─────────────────────┘  │
└───────────────────────────┘
    │
    ▼
RMSNorm（最终归一化）
    │
    ▼
output (Linear → vocab_size)          # 输出投影，与 embedding 权重共享
    │
    ▼
logits / cross_entropy loss
```
#### 各组件解读
##### 1. 词嵌入 + dropout
```python
self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)
self.dropout = nn.Dropout(args.dropout)
```
把 token id `(B,T)` 映射为 `(B,T,dim)` 的向量。
##### 2. N 层 Decoder 堆叠
```python
self.layers = torch.nn.ModuleList()
for layer_id in range(args.n_layers):
    self.layers.append(DecoderLayer(layer_id, args))
```
每层 = Pre-Norm Attention（含 GQA + RoPE）+ Pre-Norm MLP，各带残差连接。用 `ModuleList` 保证所有子模块被正确注册（参数随 `.parameters()` 返回）。
##### 3. 最终 RMSNorm + 输出投影
```python
self.norm = RMSNorm(args.dim, eps=args.norm_eps)
self.output = nn.Linear(args.dim, args.vocab_size, bias=False)
```
与 GPT-2 等 Post-Norm 原版 Transformer 不同，LLaMA 在**最后一层输出前**也做一次 norm。
##### 4. 权重共享（Weight Tying）
```python
self.tok_embeddings.weight = self.output.weight
```
输入 embedding 矩阵与输出投影矩阵共用同一参数（形状都是 `(vocab_size, dim)`）。节省 `vocab_size × dim` 个参数（如 32000×4096 ≈ 1.3 亿），且经验上对小型模型有正则化效果。
##### 5. RoPE 频率预计算 + buffer 注册
```python
freqs_cos, freqs_sin = precompute_freqs_cis(...)
self.register_buffer("freqs_cos", freqs_cos, persistent=False)
```
- 频率表只依赖位置和 head_dim，训练中不变，所以预先算好存为 buffer
- `persistent=False`：不写入 state_dict，保存 checkpoint 时不冗余存储
- forward 时按当前 `seqlen` 切片使用
##### 6. 初始化策略（两处细节）
**基础初始化**：所有 Linear / Embedding 用 N(0, 0.02)，bias 置零。
**残差分支缩放初始化**（GPT-2 的做法）：
```python
if pn.endswith('w3.weight') or pn.endswith('wo.weight'):
    torch.nn.init.normal_(p, std=0.02 / math.sqrt(2 * args.n_layers))
```
对每层残差路径上的输出投影（MLP 的 `w3`、Attention 的 `wo`）用**随深度缩小**的标准差初始化。层数越多缩得越小，防止深层网络前向传播时残差逐层累加导致方差爆炸、训练初期不稳定。
##### 7. forward：训练/推理两条路径
```python
if targets is not None:
    # 训练：输出所有位置的 logits，计算逐 token 交叉熵
    logits = self.output(h)
    self.last_loss = F.cross_entropy(..., ignore_index=0, reduction='none')
else:
    # 推理：只算最后一个位置
    logits = self.output(h[:, [-1], :])
```
- **训练**：`(B,T,vocab)` 全量 logits + 逐 token loss（`reduction='none'` 返回每个位置的 loss，方便上层用 mask 过滤 padding）
- **推理**：只取 `h[:, -1, :]` 投影，因为自回归生成只需要最后一个位置的分布，省大量计算
- 兼容 HuggingFace 风格：接受 `input_ids` / `labels` 作为别名
- 返回 `CausalLMOutputWithPast`（复用同一个容器对象省内存）
##### 8. generate：自回归采样解码
```python
@torch.inference_mode()
def generate(self, idx, ..., temperature=1.0, top_k=None):
```
逐 token 循环生成：
1. **上下文截断**：超过 `max_seq_len` 时只保留最近窗口
2. **温度缩放**：`logits / temperature`，温度→0 退化为贪婪解码（topk 取 argmax）
3. **top-k 过滤**：只保留概率最高的 k 个 token，其余置 `-Inf`
4. **softmax + 多项式采样**：`torch.multinomial` 按概率随机抽下一个 token
5. **停止条件**：生成 `stop_id` 或达到 `max_new_tokens`
6. `@torch.inference_mode()` 禁用梯度追踪和版本计数，推理更快更省内存
注意注释里说明：**未使用 KV Cache**——每步都对整个序列重新前向，是教学简化（生产实现会把 K/V 缓存下来，只算新 token）。
#### 与标准 LLaMA2 的对应关系
| 本代码 | LLaMA2 原版 | 说明 |
|---|---|---|
| `DecoderLayer` | `TransformerBlock` | Pre-Norm + 残差 × 2 子层 |
| `Attention`（GQA）| `Attention` | 70B 用 GQA，7B/13B 实际是 MHA |
| `MLP`（SwiGLU）| `FeedForward` | w1/w2/w3 三投影 |
| `RMSNorm` | `RMSNorm` | 无 bias、无 mean 中心化 |
| `output` 权重共享 | 原版**不共享** | 这是本实现的差异点 |
| RoPE | RoPE | 相同 |
#### 一句话总结
这份代码实现了一个**decoder-only 的 LLaMA2**：token 嵌入 → N 层「RMSNorm + GQA/RoPE 注意力 + RMSNorm + SwiGLU MLP」（均带残差）→ 最终 RMSNorm → 输出投影（与 embedding 权重共享），训练时全序列算逐 token 交叉熵，推理时走温度/top-k 采样自回归生成，并用预计算 buffer、残差分支缩放初始化等技巧保证训练稳定与推理高效。

## 5.2 训练 Tokenizer

三种粒度：

| 类型 | 切分单位 | 词表大小 | 特点 |
| --- | --- | --- | --- |
| Word-based | 空格/标点分词 | 大 | 简单，但未登录词多、词表膨胀 |
| Character-based | 单个字符 | 小 | 无未登录词，但序列长、语义粒度细 |
| Subword | 子词 | 适中 | 平衡词表与序列长度，主流选择 |

常见 Subword 算法：

- **BPE**：从字符开始，反复合并最高频的相邻符号对；RoBERTa、GPT 系使用。
- **WordPiece**：合并能最大提升语料似然的子词；BERT 使用。
- **Unigram**：从大词表开始，逐步删除对似然影响最小的子词；T5、ALBERT 使用。

下面训练一个带聊天模板的 BPE Tokenizer。

In [20]:
import random
import json
import os
from transformers import AutoTokenizer, PreTrainedTokenizerFast
from tokenizers import (
    decoders,
    models,
    pre_tokenizers,
    trainers,
    Tokenizer,
)
from tokenizers.normalizers import NFKC
from typing import Generator

In [21]:
def read_texts_from_jsonl(file_path: str) -> Generator[str, None, None]:
    """读取JSONL文件并安全提取文本数据"""
    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            try:
                data = json.loads(line)
                if 'text' not in data:
                    raise KeyError(f"Missing 'text' field in line {line_num}")
                yield data['text']
            except json.JSONDecodeError:
                print(f"Error decoding JSON in line {line_num}")
                continue
            except KeyError as e:
                print(e)
                continue

### 创建 Tokenizer 配置

注意：`code/train_tokenizer.py` 与 5.2 节示例使用 `add_prefix_space=False`；本章 5.3 节的示例片段中出现了 `True`，两处并不一致。实际训练建议以 `code/` 中的脚本为准。

In [22]:
def create_tokenizer_config(save_dir: str) -> None:
    """创建完整的tokenizer配置文件"""
    config = {
        "add_bos_token": False,
        "add_eos_token": False,
        "add_prefix_space": False,
        "bos_token": "<|im_start|>",
        "eos_token": "<|im_end|>",
        "pad_token": "<|im_end|>",
        "unk_token": "<unk>",
        "model_max_length": 1000000000000000019884624838656,
        "clean_up_tokenization_spaces": False,
        "tokenizer_class": "PreTrainedTokenizerFast",
        "chat_template": (
            "{% for message in messages %}"
            "{% if message['role'] == 'system' %}"
            "<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
            "{% elif message['role'] == 'user' %}"
            "<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
            "{% elif message['role'] == 'assistant' %}"
            "<|im_start|>assistant\n{{ message['content'] }}<|im_end|>\n"
            "{% endif %}"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
            "{{ '<|im_start|>assistant\n' }}"
            "{% endif %}"
        )
    }

    # 保存主配置文件
    with open(os.path.join(save_dir, "tokenizer_config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=4)

    # 创建special_tokens_map.json
    special_tokens_map = {
        "bos_token": "<|im_start|>",
        "eos_token": "<|im_end|>",
        "unk_token": "<unk>",
        "pad_token": "<|im_end|>",
        "additional_special_tokens": ["<s>", "</s>"]
    }
    with open(os.path.join(save_dir, "special_tokens_map.json"), "w", encoding="utf-8") as f:
        json.dump(special_tokens_map, f, ensure_ascii=False, indent=4)

In [23]:
def train_tokenizer(data_path: str, save_dir: str, vocab_size: int = 8192) -> None:
    """训练并保存自定义tokenizer"""
    os.makedirs(save_dir, exist_ok=True)
    
    # 初始化tokenizer
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer = NFKC()  # 添加文本规范化
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()

    # 配置特殊token
    special_tokens = [
        "<unk>", 
        "<s>", 
        "</s>", 
        "<|im_start|>", 
        "<|im_end|>"
    ]

    # 配置训练器
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        min_frequency=2,  # 提高低频词过滤
        show_progress=True,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
    )

    # 训练tokenizer
    print(f"Training tokenizer with data from {data_path}")
    texts = read_texts_from_jsonl(data_path)
    tokenizer.train_from_iterator(texts, trainer=trainer, length=os.path.getsize(data_path))

    # 验证特殊token映射
    try:
        assert tokenizer.token_to_id("<unk>") == 0
        assert tokenizer.token_to_id("<s>") == 1
        assert tokenizer.token_to_id("</s>") == 2
        assert tokenizer.token_to_id("<|im_start|>") == 3
        assert tokenizer.token_to_id("<|im_end|>") == 4
    except AssertionError as e:
        print("Special tokens mapping error:", e)
        raise

    # 保存tokenizer文件
    tokenizer.save(os.path.join(save_dir, "tokenizer.json"))
    
    # 创建配置文件
    create_tokenizer_config(save_dir)
    print(f"Tokenizer saved to {save_dir}")

#### `train_tokenizer` 逐行解读：为什么要这样设置？
```
这个函数用 HuggingFace `tokenizers` 库训练一个 **Byte-Level BPE 分词器**（与 GPT-2 / LLaMA 同款方案）。下面逐行拆解每一处设计的动机。
```
---
##### 1. 函数签名
```python
def train_tokenizer(data_path: str, save_dir: str, vocab_size: int = 8192) -> None:
```
- `data_path`：JSONL 格式的训练语料
- `save_dir`：tokenizer 保存目录
- `vocab_size=8192`：词表大小。教学/小模型场景取小词表（LLaMA2 是 32000），**词表越小，embedding 和输出投影的参数越少**（权重共享后节省 `vocab_size × dim`），小数据集也更容易把每个 token 训充分——避免大量训练中出现次数极少的“死 token”。
```python
os.makedirs(save_dir, exist_ok=True)
# 目录不存在则创建，已存在不报错，保证保存路径可用。
```

---
##### 2. 初始化 BPE 模型
```python
tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
```
- 选择 **BPE（Byte Pair Encoding）** 算法：从字符开始，迭代合并语料中出现最频繁的相邻 token 对，逐步长出整个词表
- `unk_token="<unk>"`：遇到无法编码的内容时的兜底符号。⚠️ 注意：Byte-Level 方案下理论上**任何字节都能编码**，`<unk>` 几乎不会被触发，这里更多是接口约定和保险
---
##### 3. 文本规范化
```python
tokenizer.normalizer = NFKC()
```
- NFKC（Normalization Form KC）：Unicode 规范化，把“视觉等价”的不同编码统一
- 例子：全角数字 `１２３` → `123`；组合字符 `é`（e + ́）→ 预组合的 `é`
- **目的**：同一内容只有一种表示，BPE 统计合并频率时不会被编码差异稀释，词表利用率更高
---
##### 4. 预分词器（关键设计）
```python
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
```
这是整个配置中最核心的一行，做了两件事：
###### (a) Byte-Level：字节回退编码
文本先按 UTF-8 转成字节，每个字节映射到 256 个可打印 Unicode 字符（如 `空格→Ġ`）。好处：
- **零 OOV**：任何输入（包括 emoji、代码、生僻字）都能表示为字节序列，永远不会解码失败
- 词表基础是 256 字节而非几万字符，小词表也能覆盖所有文本
###### (b) GPT-2 风格的字节级预切分
在字节化之前先按正则把文本切成“单词块”（字母串 / 数字串 / 标点 / 空格），BPE 的合并**不会跨越块边界**。这样：
- `dog` 和 `dog!` 中的 `dog` 会被编码为同一个 token，词表复用率高
- 模型学到的是“词内子词”规律，而非混乱的字节序列
###### `add_prefix_space=False` GPT-2 默认会在每个词前加空格（`add_prefix_space=True`）。这里关闭，即**不擅自添加前导空格**，token 边界严格跟随原文——对中文等多空格敏感度低的语料更干净，也避免改变原文内容。
---
##### 5. 解码器
```python
tokenizer.decoder = decoders.ByteLevel()
```
```
与预分词器配套的**逆向操作**：解码时把字节字符还原成真正的字节再 UTF-8 解码、正确处理空格拼接（去掉 `Ġ` 的标记语义）。编码/解码必须成对配置，否则 `decode` 出来会是 `ĠhelloĠworld` 这样的乱码。
```
---
##### 6. 特殊 token
```python
special_tokens = [
    "<unk>",        # id=0 未知兜底
    "<s>",          # id=1 句子开始 (BOS)
    "</s>",         # id=2 句子结束 (EOS)
    "<|im_start|>", # id=3 对话轮开始（ChatML 风格）
    "<|im_end|>",   # id=4 对话轮结束
]
```
设计动机：
- **必须显式声明**：特殊 token 加入后，BPE 训练器**不会把它们拆成子词**，保证 `tokenizer.encode("<s>")` 永远得到单个 id 1，而不是被切碎
- **顺序决定 id**：`special_tokens` 列表顺序就是分配的 id 顺序，放在词表最前面（0~4）
- **`<|im_start|>/<|im_end|>`**：ChatML 对话模板标记，用于 SFT 阶段构造多轮对话数据、训练时用 `im_end` 作为停止符、generate 时作为 `stop_id`
- **id 0 是 `<unk>`**：与后面 Transformer 里的 `ignore_index=0` 呼应——loss 计算跳过 id 0（padding 用 0 填充时不参与训练）
---
##### 7. BPE 训练器配置
```python
trainer = trainers.BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=special_tokens,
    min_frequency=2,
    show_progress=True,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)
```
| 参数 | 作用 | 为什么这样设 |
|---|---|---|
| `vocab_size=8192` | 目标词表上限（含 5 个特殊 token） | 小模型 + 小语料的平衡点 |
| `special_tokens=...` | 特殊 token 优先占据 id 0~4 | 见上节 |
| `min_frequency=2` | 出现少于 2 次的合并对不做合并 | 过滤噪声/错别字等极端低频对，防止词表被“只出现一次”的垃圾 token 占据，8192 个坑位很宝贵 |
| `show_progress=True` | 显示训练进度条 | 语料大时便于观察 |
| `initial_alphabet=ByteLevel.alphabet()` | **强制把全部 256 个字节字符放进初始词表** | 关键：若某字节在语料中出现极少，BPE 可能不会把它纳入词表；强制加入保证“任何输入都能编码”的 Byte-Level 承诺成立 |
---
##### 8. 执行训练
```python
texts = read_texts_from_jsonl(data_path)
tokenizer.train_from_iterator(texts, trainer=trainer, length=os.path.getsize(data_path))
```
- `read_texts_from_jsonl`：生成器逐条读出文本，**不一次性载入内存**，配合 `train_from_iterator` 可处理超大语料
- `train_from_iterator`：流式训练接口（区别于 `train(files)`，后者只吃纯文本文件，无法处理 JSONL 结构）
- `length=os.path.getsize(data_path)`：告知预计样本量，让进度条显示正确百分比（纯优化体验，不影响结果）
训练过程：以 256 字节 + 5 特殊 token 为起点，反复统计语料中最高频的相邻 token 对并合并，直到填满 8192 个词表位置。
---
##### 9. 验证特殊 token 映射
```python
assert tokenizer.token_to_id("<unk>") == 0
assert tokenizer.token_to_id("<s>") == 1
...
```
**训练完成后立刻断言**id 与预期一致。为什么必要？
- 特殊 token 的 id 是下游一切逻辑的**隐式契约**：SFT 模板写死 `<|im_start|>` 的 id、loss 的 `ignore_index=0`、generate 的 `stop_id` 都依赖它
- 若因库版本差异、`min_frequency` 或语料异常导致 id 错位，训练出的模型会静默出错（比如把 `im_end` 当普通文本），**越早暴露越好**
- 用 `try/except` 捕获并打印明确错误信息后 `raise`，避免吞掉异常
---
##### 10. 保存
```python
tokenizer.save(os.path.join(save_dir, "tokenizer.json"))
create_tokenizer_config(save_dir)
```
- `tokenizer.json`：单文件保存**全部词表 + 合并规则 + normalizer/pre_tokenizer/decoder 配置**，之后 `Tokenizer.from_file()` 一行即可复原，无需重新训练
- `create_tokenizer_config`：额外生成 HuggingFace 风格配置（`tokenizer_config.json`、`special_tokens_map.json` 等），方便与 `transformers` 生态（`AutoTokenizer`）对接
---
##### 整体设计思想总结
| 设计点 | 动机 |
|---|---|
| **Byte-Level BPE** | 零 OOV、小词表也能覆盖一切输入、与 GPT-2/LLaMA 技术路线一致 |
| **NFKC + ByteLevel 预切分** | 统一表示提高统计质量；词内合并提高 token 复用率 |
| **小词表 8192 + min_frequency=2** | 参数省、token 训练充分、过滤噪声 |
| **5 个特殊 token 固定 id 0~4** | 支撑对话模板（ChatML）、loss mask、停止条件等下游契约 |
| **强制初始字节字母表** | 保证任意输入可编码的硬承诺 |
| **断言验证 + 流式训练 + 单文件保存** | 防静默错误、省内存、易复用 |
一句话：这是一套为**小规模从零训练对话模型**定制的 tokenizer 流水线——用 Byte-Level BPE 保证鲁棒性，用小词表匹配小模型容量，用固定的特殊 token id 支撑后续 SFT/推理全流程，并用断言在训练当场锁死这些契约。

In [25]:
def eval_tokenizer(tokenizer_path: str) -> None:
    """评估tokenizer功能"""
    try:
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    except Exception as e:
        print(f"Error loading tokenizer: {e}")
        return

    # 测试基本属性
    print("\n=== Tokenizer基本信息 ===")
    print(f"Vocab size: {len(tokenizer)}")
    print(f"Special tokens: {tokenizer.all_special_tokens}")
    print(f"Special token IDs: {tokenizer.all_special_ids}")

    # 测试聊天模板
    messages = [
        {"role": "system", "content": "你是一个AI助手。"},
        {"role": "user", "content": "How are you?"},
        {"role": "assistant", "content": "I'm fine, thank you. and you?"},
        {"role": "user", "content": "I'm good too."},
        {"role": "assistant", "content": "That's great to hear!"},
    ]
    
    print("\n=== 聊天模板测试 ===")
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        # add_generation_prompt=True
    )
    print("Generated prompt:\n", prompt, sep="")

    # 测试编码解码
    print("\n=== 编码解码测试 ===")
    encoded = tokenizer(prompt, truncation=True, max_length=256)
    decoded = tokenizer.decode(encoded["input_ids"], skip_special_tokens=False)
    print("Decoded text matches original:", decoded == prompt)

    # 测试特殊token处理
    print("\n=== 特殊token处理 ===")
    test_text = "<|im_start|>user\nHello<|im_end|>"
    encoded = tokenizer(test_text).input_ids
    decoded = tokenizer.decode(encoded)
    print(f"Original: {test_text}")
    print(f"Decoded:  {decoded}")
    print("Special tokens preserved:", decoded == test_text)

In [ ]:
# 训练完成后取消注释，传入实际保存目录
# eval_tokenizer("tokenizer_k")

输出示例（来自原文）：

```text
=== Tokenizer基本信息 ===
Vocab size: 6144
Special tokens: ['<|im_start|>', '<|im_end|>', '<unk>', '<s>', '</s>']
Special token IDs: [3, 4, 0, 1, 2]

=== 编码解码测试 ===
Decoded text matches original: False
```

当 `add_prefix_space` 或特殊 token 的预切分方式不一致时，解码结果可能出现多余空格，可通过对比 `tokenizer.decode(tokenizer.encode(text))` 定位。

## 5.3 预训练一个小型 LLM

整体流程：

```text
下载语料 → 清洗/切块 → 训练 Tokenizer → 构建 Dataset
        → 预训练（CLM）→ SFT（指令微调）→ 采样生成
```

> 下面数据下载与处理会读写 GB 级文件，命令以注释形式保留，避免误执行。

### 5.3.1 数据下载与预处理

```bash
# 下载预训练数据集
modelscope download --dataset ddzhu123/seq-monkey \
  mobvoi_seq_monkey_general_open_corpus.jsonl.tar.bz2 --local_dir your_local_dir

# 解压
tar -xvf your_local_dir/mobvoi_seq_monkey_general_open_corpus.jsonl.tar.bz2

# 下载 SFT 数据集
huggingface-cli download --repo-type dataset --resume-download \
  BelleGroup/train_3.5M_CN --local-dir BelleGroup
```

In [26]:
import json
from tqdm import tqdm


def split_text(text, chunk_size=512):
    """将长文本按字符数切块，每一块作为一条预训练样本。"""
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]


def convert_message(data):
    """把 BelleGroup 的 from/value 格式转换为 role/content 对话格式。"""
    message = [
        {"role": "system", "content": "你是一个AI助手"},
    ]
    for item in data:
        if item['from'] == 'human':
            message.append({'role': 'user', 'content': item['value']})
        elif item['from'] == 'assistant':
            message.append({'role': 'assistant', 'content': item['value']})
    return message


# 原文的处理循环（按需取消注释后运行）：
# with open('seq_monkey_datawhale.jsonl', 'a', encoding='utf-8') as pretrain:
#     with open('mobvoi_seq_monkey_general_open_corpus.jsonl', 'r', encoding='utf-8') as f:
#         for line in tqdm(f.readlines(), desc='Processing pretrain data'):
#             text = json.loads(line)['text']
#             for chunk in split_text(text):
#                 pretrain.write(json.dumps({'text': chunk}, ensure_ascii=False) + '\n')
#
# with open('BelleGroup_sft.jsonl', 'a', encoding='utf-8') as sft:
#     with open('BelleGroup/train_3.5M_CN.json', 'r', encoding='utf-8') as f:
#         for item in tqdm(f.readlines(), desc='Processing SFT data', unit='lines'):
#             message = convert_message(json.loads(item)['conversations'])
#             sft.write(json.dumps(message, ensure_ascii=False) + '\n')

### 5.3.2 训练 Tokenizer

5.3 节与 5.2.4 节的 Tokenizer 代码基本相同，只是训练参数不同：预训练使用 `vocab_size=6144`。完整脚本见 `code/train_tokenizer.py`。

In [27]:
# 与 5.2 的 train_tokenizer 相同，只是词表大小改为 6144
TOKENIZER_DIR = "tokenizer_k"
VOCAB_SIZE = 6144

# 准备好 seq_monkey_datawhale.jsonl 后取消注释：
# train_tokenizer("seq_monkey_datawhale.jsonl", TOKENIZER_DIR, VOCAB_SIZE)
# eval_tokenizer(TOKENIZER_DIR)

### 5.3.3 Dataset

预训练与 SFT 都需要把文本转成 `(X, Y, loss_mask)`：

- `X`：输入序列，`input_ids[:-1]`；
- `Y`：预测目标，`input_ids[1:]`，与 `X` 错开一位；
- `loss_mask`：标记哪些位置参与损失计算，padding 位置为 0。

预训练 Dataset 通过预计算每行的字节偏移，避免每次 `__getitem__` 都从头扫描整个文件。

In [28]:
import json
import numpy as np
import torch

from torch.utils.data import Dataset

class PretrainDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=512):
        super().__init__()
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.padding = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
        # 预计算每行的起始字节偏移量
        self._offsets = []
        with open(data_path, 'rb') as f:
            self._offsets.append(0)
            while f.readline():
                self._offsets.append(f.tell())
        self._total_lines = len(self._offsets) - 1  # 最后一个 tell() 是 EOF

    def __len__(self):
        return self._total_lines

    def __getitem__(self, index: int):
        with open(self.data_path, 'rb') as f:
            f.seek(self._offsets[index])
            line = f.readline().decode('utf-8')
        sample = json.loads(line)
        text = f"{self.tokenizer.bos_token}{sample['text']}"
        input_id = self.tokenizer(text).data['input_ids'][:self.max_length]
        text_len = len(input_id)
        # 没满最大长度的剩余部分
        padding_len = self.max_length - text_len
        input_id = input_id + [self.padding] * padding_len
        # 0表示不计算损失
        loss_mask = [1] * text_len + [0] * padding_len

        input_id = np.array(input_id)
        X = np.array(input_id[:-1]).astype(np.int64)
        Y = np.array(input_id[1:]).astype(np.int64)
        loss_mask = np.array(loss_mask[1:]).astype(np.int64)
        return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(loss_mask)

#### `PretrainDataset` 详解：预训练数据集的构建逻辑。这个类实现了一个**内存友好的 JSONL 预训练数据集**：每行 JSON 文本为一个样本，负责 tokenize、padding、构造 next-token 预测的 `(X, Y, loss_mask)` 三元组。逐块拆解：
---
##### 1. 初始化：基本配置
```python
def __init__(self, data_path, tokenizer, max_length=512):
    self.data_path = data_path
    self.tokenizer = tokenizer
    self.max_length = max_length
    self.padding = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
```
- `max_length=512`：每条样本截断/填充到固定长度——预训练需要定长序列才能拼 batch
- `padding`：填充符优先用 tokenizer 的 `pad_token_id`，没有则回退为 0。结合前面 tokenizer 的设计（id 0 = `<unk>`，且模型 loss 的 `ignore_index=0`），**填充位置不会被计入损失**，这是整条数据链路的隐式契约
---
##### 2. 核心技巧：预计算行偏移量（字节索引）
```python
self._offsets = []
with open(data_path, 'rb') as f:
    self._offsets.append(0)
    while f.readline():
        self._offsets.append(f.tell())
self._total_lines = len(self._offsets) - 1
```
这段做了什么：
- **二进制模式**逐行扫一遍文件，`f.tell()` 记录每行**结束时的字节位置**
- `_offsets` 变成 `[第0行起始, 第1行起始, 第2行起始, ... , EOF位置]`，所以总行数 = 长度 − 1
举例，文件内容为 3 行（每行长度不等）：
```
offsets = [0, 25, 63, 100]
           ↑    ↑   ↑    ↑
          行0  行1  行2  EOF
_total_lines = 3
```
**为什么要这样做？**
- JSONL 预训练语料往往几 GB 甚至几十 GB，**不能整体载入内存**
- `Dataset` 只需要支持**随机访问** `__getitem__(i)`——有了偏移量表，取第 i 条样本时直接：
```python
f.seek(self._offsets[index])   # 跳到第 i 行开头（O(1)）
line = f.readline()            # 只读这一行
```
- 不用每次从文件头逐行数到第 i 行（那是 O(i) 的代价），随机抽样、DataLoader 多 worker 打乱读取都高效
- 只需在初始化时做一次线性扫描（很快，只按行跳过不解析），换取之后的随机 O(1) 访问
---
##### 3. `__len__` 和 `__getitem__`
```python
def __len__(self):
    return self._total_lines

# PyTorch `DataLoader` 靠这个知道数据集大小、划分 batch。
```
  
---
##### 4. 读取与拼接 BOS
```python
with open(self.data_path, 'rb') as f:
    f.seek(self._offsets[index])
    line = f.readline().decode('utf-8')
    sample = json.loads(line)
    text = f"{self.tokenizer.bos_token}{sample['text']}"
```
- 每行 JSON 形如 `{"text": "..."}`，取出文本字段
- **手动在开头拼上 `<s>`（BOS）**：告诉模型“这是一个独立文档/样本的开始”，让模型学习文档起始的分布；也与生成时从 BOS 起步的推理流程对齐
---
##### 5. 编码 + 截断 + 填充
```python
input_id = self.tokenizer(text).data['input_ids'][:self.max_length]
text_len = len(input_id)
padding_len = self.max_length - text_len
input_id = input_id + [self.padding] * padding_len
```
- tokenize 后超过 512 截断；不足则补 pad 到定长
```
例（max_length=8）:  [<s>, A, B, C]  →  [<s>, A, B, C, 0, 0, 0, 0]
```
---
##### 6. loss_mask：标记哪些位置算损失
```python
loss_mask = [1] * text_len + [0] * padding_len
```
- 真实 token 位置 → 1；填充位置 → 0
- 后续训练中 `loss = (loss * loss_mask).sum()` 即可**屏蔽 padding**，不让模型在填充符上浪费学习信号
---
##### 7. 构造 next-token 预测对（关键）
```python
X = np.array(input_id[:-1]).astype(np.int64)          # 输入
Y = np.array(input_id[1:]).astype(np.int64)           # 标签
loss_mask = np.array(loss_mask[1:]).astype(np.int64)  # mask 也同步错位
```
这是**语言模型自监督的核心构造**：把同一序列错开一位，得到“输入 → 应预测的下一个 token"：
```
input_id:  [<s>   A     B     C    0    0    0    0]
X:         [<s>   A     B     C    0    0    0  ]   ← 去掉最后一个
Y:         [ A    B     C     0    0    0    0  ]   ← 去掉第一个（整体左移一位）
loss_mask: [ 1    1     1     1    0    0    0  ]   ← 同样去掉第一个
对应关系:  X[0]=<s> → 预测 Y[0]=A
          X[1]=A   → 预测 Y[1]=B
          ...
          X[6]=0   → 预测 Y[6]=0（mask=0，不算损失）
```
```
**为什么去掉第一个 pad 的 mask 位？**
注意 `loss_mask` 也要左移一位：mask 里第 i 位对应的是 `X[i] → Y[i]` 这个预测任务的权重。原始 mask 的第一个 1 对应"token `<s>` 本身"，但 `<s>` 作为第 0 个输入没有对应的“被预测”身份，错位后统一对齐。padding 的 mask=0 起始位置也同步左移，保证 mask 与 `(X→Y)` 对严格对应。
**为什么 `astype(np.int64)`？**
embedding 层要求整数索引类型为 int64（`nn.Embedding` 默认权重是 float32，索引必须是 Long），显式转换避免类型不匹配报错。
```
---
##### 8. 返回张量
```python
return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(loss_mask)

DataLoader 会自动把各样本堆叠成 batch：`(B, 511)` 的 X、Y、loss_mask。
```

---
##### 整体流程图
```
JSONL 文件（大）
   │ 初始化时: 扫描记录每行字节偏移 → 随机 O(1) 定位
   ▼
__getitem__(i): seek + 读一行 → json 解析
   ▼
<s> + text → tokenizer → 截断到 512 → padding 到定长
   ▼
构造 (X = 序列[:-1],  Y = 序列[1:],  loss_mask 左移对齐)
   ▼
返回三个 torch 张量 → 模型计算逐 token 交叉熵，mask 屏蔽 pad
```
##### 设计要点总结
| 设计 | 动机 |
|---|---|
| 字节偏移表 + `seek` | 大文件下随机访问，不全量载入内存 |
| 拼 BOS | 显式标记样本起点，与生成流程一致 |
| 截断 + padding 到定长 | 可拼 batch |
| X/Y 错一位 | 实现 next-token prediction 自监督 |
| loss_mask 左移一位并屏蔽 pad | 只在真实 token 上计算损失 |
| `int64` 转换 | 满足 embedding 的索引类型要求 |
一句话：这是一个**面向大 JSONL 语料的流式预训练 Dataset**——用偏移量索引实现省内存的随机访问，把每条文本加工成「定长的、带 BOS 的、错位对齐的 (输入，标签，损失掩码) 三元组」，直接喂给 causal LM 做 next-token 训练。

In [33]:
class SFTDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=512):
        super().__init__()
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.padding = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
        self._offsets = []
        with open(data_path, 'rb') as f:
            self._offsets.append(0)
            while f.readline():
                self._offsets.append(f.tell())
        self._total_lines = len(self._offsets) - 1 

    def __len__(self):
        return self._total_lines

    def generate_loss_mask(self, input_ids):
        # 生成 loss mask, 0 表示不计算损失, 1 表示计算损失
        mask = [0] * len(input_ids)
        a_sequence = self.tokenizer("<|im_start|>assistant\n")['input_ids']  # <|im_start|>assistant\n
        a_length = len(a_sequence)
        n = len(input_ids)
        i = 0
        
        while i <= n - a_length:
            # 检查当前位置是否匹配目标子序列
            match = True
            for k in range(a_length):
                if input_ids[i + k] != a_sequence[k]:
                    match = False
                    break
            if match:
                # 从子序列结束的位置开始查找第一个 4 (eos_token_id)
                j = None
                for idx in range(i + a_length, n):
                    if input_ids[idx] == self.tokenizer.eos_token_id:
                        j = idx
                        break
                if j is not None:
                    start = i + a_length
                    end = j  # 结束位置设为j（包含4）
                    # 标记区间为1（包括start到end）
                    if start <= end:
                        for pos in range(start, end + 1):
                            if pos < len(mask):
                                mask[pos] = 1
                # 跳过当前子序列，避免重叠匹配
                i += a_length
            else:
                i += 1
        return mask

    def __getitem__(self, index: int):
        with open(self.data_path, 'rb') as f:
            f.seek(self._offsets[index])
            line = f.readline().decode('utf-8')
        sample = json.loads(line)
        text = self.tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=False)
        input_id = self.tokenizer(text).data['input_ids'][:self.max_length]
        text_len = len(input_id)
        # 没满最大长度的剩余部分
        padding_len = self.max_length - text_len
        input_id = input_id + [self.padding] * padding_len
        # 0表示不计算损失
        loss_mask = self.generate_loss_mask(input_id)

        input_id = np.array(input_id)
        X = np.array(input_id[:-1]).astype(np.int64)
        Y = np.array(input_id[1:]).astype(np.int64)
        loss_mask = np.array(loss_mask[1:]).astype(np.int64)
        return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(loss_mask)

SFT Dataset 的关键在 `generate_loss_mask`：

- 先定位 `<|im_start|>assistant\n` 这段 token；
- 只对 assistant 回复部分（直到 `eos_token_id`）置 1；
- system、user 以及 padding 部分置 0，不计算损失。

这样模型只学习“如何回答”，而不是拟合用户问题本身。

#### `SFTDataset` 详解：SFT 数据集——只在 assistant 回复上计算损失
这个类与之前的 `PretrainDataset` 结构几乎相同（偏移量索引、BOS... 等等），但有**两处关键区别**：
1. 用 `apply_chat_template` 把多轮对话 JSON 渲染成带 `<|im_start|>` / `<|im_end|>` 标记的文本
2. **不再简单地"mask 掉 padding"**，而是用 `generate_loss_mask` 精确找到 assistant 回复区间——**只有模型要学的回答部分才算损失**，提问和模板部分不算
这是 SFT（监督微调）区别于预训练的核心。
---
##### 1. 整体流程
```
JSONL 每行 = 一段对话，如:
[{"role":"user","content":"你好"},{"role":"assistant","content":"你好！有什么可以帮你？"}]
        │
        ▼ apply_chat_template
"<|im_start|>user\n你好<|im_end|>\n<|im_start|>assistant\n你好！有什么可以帮你？<|im_end|>"
        │
        ▼ tokenize + 截断 + padding（与 PretrainDataset 相同）
input_ids
        │
        ▼ generate_loss_mask（本类的核心新增）
只在 assistant 回复内容（含结束符）上标 1，其余全 0
        │
        ▼ X/Y 错位构造（与 PretrainDataset 相同）
(X, Y, loss_mask)
```
---
##### 2. `__getitem__`：对话模板渲染
```python
sample = json.loads(line)
text = self.tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=False)
```
- `sample` 是 messages 列表（多轮 user/assistant 交替）
- `apply_chat_template` 按模板自动拼接特殊 token；`tokenize=False` 只返回字符串（先渲染，后面统一 tokenize）
- `add_generation_prompt=False`：不在末尾追加 `<|im_start|>assistant\n` 提示（那是推理时让模型续写用的；训练时每轮回复都已在文本里）
渲染结果示例：
```
<|im_start|>user
你好<|im_end|>
<|im_start|>assistant
你好！有什么可以帮你？<|im_end|>
```
```
后续的截断、padding 与 PretrainDataset 完全一样（定长 512，pad 用 id 0）。
```
---
##### 3. `generate_loss_mask`：本类的灵魂
目标：把 mask 中**只有“assistant 说的话”区间置 1**，其他（system/user 消息、模板标记、padding）全为 0。
###### 第一步：锚点序列
```python
a_sequence = self.tokenizer("<|im_start|>assistant\n")['input_ids']
a_length = len(a_sequence)
```
`<|im_start|>assistant\n` 是每轮回复的**开头标记**。它在 input_ids 里是固定的一小段 token 序列（比如 `[3, X, Y, Z]`）。用它作为“回复开始”的定位锚点。
###### 第二步：滑动窗口暴力匹配
```python
while i <= n - a_length:
    match = True
    for k in range(a_length):
        if input_ids[i + k] != a_sequence[k]:
            match = False
            break
```
在序列上逐位置 i 检查：从 i 开始的 `a_length` 个 token 是否与锚点序列完全一致。找到即定位了一轮回复的起点。
###### 第三步：找回复终点（第一个 eos）
```python
j = None
for idx in range(i + a_length, n):
    if input_ids[idx] == self.tokenizer.eos_token_id:
        j = idx
        break
```
从锚点结束后向后扫描，找第一个 **eos（`<|im_end|>`，id=4）**——即本轮回复的结束标记。
###### 第四步：标记区间并跳过
```python
start = i + a_length      # 锚点之后 = 回复正文开始
end = j                   # eos 的位置（包含 eos 本身）
for pos in range(start, end + 1):
    mask[pos] = 1
i += a_length             # 跳过锚点，避免重叠匹配
```
关键细节：
- **`end` 包含 eos（`end + 1`）**：让模型也学习“在回答结束时输出 `<|im_end|>`”——这正是推理时 generate 用 eos 停止的依据。若不训练 eos，模型会永远说下去
- **mask 标的是“被预测的位置”**：配合后面 `loss_mask = loss_mask[1:]` 的错位，实际含义是“预测 Y 中这些位置的 token 要算损失”，即模型要学会**生成**回复正文 + eos
- **`i += a_length`**：匹配成功后跳过整个锚点继续搜索，防止嵌套/重叠匹配；匹配失败才 `i += 1`
- 找不到 eos（文本被 512 截断）→ 该轮不标记（`j is None` 走 else）
###### Mask 效果示例
```
token:    <|im_start|> user  你好  <|im_end|>  <|im_start|> assistant  你好！  有什么... <|im_end|>  pad ...
mask:        0          0     0       0          0            1       1         1        0
                                                    └──────── 只学 assistant 回答 + 结束符 ────────┘
```
---
##### 4. X/Y 错位构造（与 PretrainDataset 相同）
```python
X = input_id[:-1];  Y = input_id[1:];  loss_mask = loss_mask[1:]
```
同样的“输入左移/标签右移一位”构造。错位后 mask 的语义变为：
```
X:    ... <|im_start|>assistant  你   好 ！  <|im_end|> ...
Y:    ... 你                     好 ！ <|im_end|>  <|im_end|>?
mask: ...              1         1     1       1
```
即：从锚点后的第一个 token 开始，**预测回复正文和 eos 的损失都被计入**；预测 user 消息、模板 token、pad 的位置 mask=0，不学。
> 对比 PretrainDataset：那边 mask 只是“屏蔽 padding”；这边 mask 是“屏蔽整个非 assistant 区域”，语义上是“只学该学的”。
---
##### 5. 为什么 SFT 要这样做？（设计动机）
预训练时全序列都算损失（学语言本身）；SFT 时如果仍全算损失，模型会被迫学习“生成用户的问题”，带来两个坏处：
1. **污染生成行为**：推理时模型续写对话，可能学会自己给自己提问、复读 user 内容
2. **浪费容量**：模板 token（`<|im_start|>user` 等）是固定格式，不需要“生成能力”
正确的目标恰恰是：**给定对话历史，只学习生成 assistant 的回答**。这就是所谓的 **loss masking / prompt loss masking**，Alpaca、ChatGLM、LLaMA 的 SFT 实现普遍采用。
---
##### 6. 与 PretrainDataset 的对比总结
| 方面 | PretrainDataset | SFTDataset |
|---|---|---|
| 数据格式 | `{"text": "..."}` 纯文本 | messages 多轮对话列表 |
| 文本构造 | 手动拼 BOS | `apply_chat_template` 渲染模板 |
| loss_mask | 规则生成：真实 token=1，pad=0 | **语义生成**：仅 assistant 回复+eos=1 |
| 学习目标 | 无差别学整段文本分布 | 只学“如何回答” |
| 代码结构 | — | 偏移量索引、截断/padding、X/Y 错位完全复用 |
##### 一句话总结
`SFTDataset` = PretrainDataset 的骨架 + ChatML 对话模板渲染 + 一个基于 `<|im_start|>assistant\n` 锚点匹配和 eos 定位的 loss mask 生成器，实现了“**只对 assistant 回复（含结束符）计算交叉熵、屏蔽提问与模板**”的 SFT 核心机制，让小模型在微调时学会“回答”而不是“复述对话”。

### 5.3.4 预训练

预训练脚本包含以下工程要点：

- **学习率调度**：线性 warmup + 余弦退火，最后保持最小学习率。
- **梯度累积**：`loss / accumulation_steps` 多次反传后再 `optimizer.step()`，用小显存模拟大 batch。
- **混合精度**：`torch.amp.autocast` + `GradScaler`，降低显存并加速。
- **损失掩码**：`loss = (loss * loss_mask).sum() / loss_mask.sum()`，只统计有效 token。
- **多卡**：书中代码用 `torch.nn.DataParallel` 包装模型；文件名为 `ddp_*`，但实际并非 `DistributedDataParallel`。
- **日志**：可选 `swanlab` 记录 loss 与学习率。

> `RUN_TRAINING` 默认为 `False`，因此运行该单元格只会定义函数，不会启动训练。

In [30]:
# ---- 训练相关导入（原脚本见 code/ddp_pretrain.py）----
import os
import platform
import argparse
import time
import warnings
import math
import pandas as pd
import torch
from torch import optim
from torch.utils.data import DataLoader
from contextlib import nullcontext
from transformers import AutoTokenizer

try:
    import swanlab
except ImportError:
    swanlab = None  # 未安装时只影响日志记录

warnings.filterwarnings('ignore')
RUN_TRAINING = False  # 改为 True 并准备好数据/GPU 后才会执行 main


def Logger(content):
    """统一日志输出，便于替换为其他日志工具。"""
    print(content)

def get_lr(it, all):
    """
    计算当前迭代的学习率，使用余弦退火调度策略
    
    学习率调度策略：
    1. Warmup阶段：学习率从0线性增长到目标学习率
    2. 余弦退火阶段：学习率按余弦函数衰减到最小学习率
    3. 超出训练步数后：保持最小学习率
    
    Args:
        it (int): 当前迭代步数
        all (int): 总迭代步数
        
    Returns:
        float: 当前步数对应的学习率
    """
    warmup_iters = args.warmup_iters  # 预热迭代次数
    lr_decay_iters = all  # 学习率衰减的总迭代次数
    min_lr = args.learning_rate / 10  # 最小学习率，为初始学习率的1/10

    # Warmup阶段：线性增长
    if it < warmup_iters:
        return args.learning_rate * it / warmup_iters
    
    # 超出训练步数：保持最小学习率
    if it > lr_decay_iters:
        return min_lr
    
    # 余弦退火阶段
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))  # 余弦系数
    return min_lr + coeff * (args.learning_rate - min_lr)

def train_epoch(epoch):
    """
    训练一个epoch的函数
    
    实现了完整的训练循环，包括：
    1. 数据加载和设备转移
    2. 动态学习率调整
    3. 前向传播和损失计算
    4. 梯度累积和反向传播
    5. 梯度裁剪和优化器更新
    6. 日志记录和模型保存
    
    Args:
        epoch (int): 当前epoch编号
    """
    start_time = time.time()  # 记录开始时间
    
    # 遍历数据加载器中的每个batch
    for step, (X, Y, loss_mask) in enumerate(train_loader):
        # 将数据转移到指定设备（GPU/CPU）
        X = X.to(args.device)  # 输入序列
        Y = Y.to(args.device)  # 目标序列
        loss_mask = loss_mask.to(args.device)  # 损失掩码，用于忽略padding token

        # 计算当前步骤的学习率
        lr = get_lr(epoch * iter_per_epoch + step, args.epochs * iter_per_epoch)
        # 更新优化器中所有参数组的学习率
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # 使用混合精度训练上下文
        with ctx:
            # 前向传播
            out = model(X, Y)
            # 计算损失并除以累积步数（用于梯度累积）
            loss = out.last_loss / args.accumulation_steps
            # 将loss_mask展平为一维
            loss_mask = loss_mask.view(-1)
            # 应用掩码计算有效损失（忽略padding位置）
            loss = torch.sum(loss * loss_mask) / loss_mask.sum()

        # 使用scaler进行混合精度的反向传播
        scaler.scale(loss).backward()

        # 每accumulation_steps步执行一次优化器更新
        if (step + 1) % args.accumulation_steps == 0:
            # 取消梯度缩放，准备梯度裁剪
            scaler.unscale_(optimizer)
            # 梯度裁剪，防止梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

            # 执行优化器步骤
            scaler.step(optimizer)
            # 更新scaler的缩放因子
            scaler.update()

            # 清零梯度，set_to_none=True可以节省内存
            optimizer.zero_grad(set_to_none=True)

        # 每log_interval步记录一次日志
        if step % args.log_interval == 0:
            spend_time = time.time() - start_time
            # 打印训练进度信息
            Logger(
                'Epoch:[{}/{}]({}/{}) loss:{:.3f} lr:{:.7f} epoch_Time:{}min;'.format(
                    epoch + 1,
                    args.epochs,
                    step,
                    iter_per_epoch,
                    loss.item() * args.accumulation_steps,  # 恢复真实的loss值
                    optimizer.param_groups[-1]['lr'],
                    spend_time / (step + 1) * iter_per_epoch // 60 - spend_time // 60))
            
            # 如果启用SwanLab，记录训练指标
            if args.use_swanlab:
                swanlab.log({
                    "loss": loss.item() * args.accumulation_steps,
                    "lr": optimizer.param_groups[-1]['lr']
                })

        # 每save_interval步保存一次模型
        if (step + 1) % args.save_interval == 0:
            model.eval()  # 切换到评估模式
            # 构建检查点文件名
            ckp = f'{args.save_dir}/pretrain_{lm_config.dim}_{lm_config.n_layers}_{lm_config.vocab_size}.pth'

            # 处理多卡保存：如果是DataParallel模型，需要访问.module属性
            state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
            torch.save(state_dict, ckp)
            model.train()  # 切换回训练模式
        
        # 每20000步保存一个带步数标记的检查点
        if (step + 1) % 20000 == 0:
            model.eval()
            # 构建带步数的检查点文件名
            ckp = f'{args.save_dir}/pretrain_{lm_config.dim}_{lm_config.n_layers}_{lm_config.vocab_size}_step{step+1}.pth'

            # 保存模型状态字典
            state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
            torch.save(state_dict, ckp)
            model.train()


def init_model():
    """
    初始化模型和分词器
    
    功能包括：
    1. 加载预训练的分词器
    2. 创建Transformer模型
    3. 设置多GPU并行训练（如果可用）
    4. 将模型移动到指定设备
    5. 统计并打印模型参数量
    
    Returns:
        tuple: (model, tokenizer) 初始化后的模型和分词器
    """
    def count_parameters(model):
        """
        统计模型中可训练参数的数量
        
        Args:
            model: PyTorch模型
            
        Returns:
            int: 可训练参数总数
        """
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 从本地路径加载预训练的分词器
    tokenizer = AutoTokenizer.from_pretrained('./tokenizer_k/')

    # 根据配置创建Transformer模型
    model = Transformer(lm_config)
    
    # 多卡初始化：检查可用GPU数量并设置DataParallel
    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        Logger(f"Using {num_gpus} GPUs with DataParallel!")
        # 使用DataParallel包装模型以支持多GPU训练
        model = torch.nn.DataParallel(model)
    
    # 将模型移动到指定设备（GPU或CPU）
    model = model.to(args.device)
    
    # 计算并打印模型参数量（以百万为单位）
    Logger(f'LLM总参数量：{count_parameters(model) / 1e6:.3f} 百万')
    return model, tokenizer


if __name__ == "__main__" and RUN_TRAINING:
    # ==================== 命令行参数解析 ====================
    parser = argparse.ArgumentParser(description="Tiny-LLM Pretraining")
    
    # 基础训练参数
    parser.add_argument("--out_dir", type=str, default="base_model_215M", help="模型输出目录")
    parser.add_argument("--epochs", type=int, default=1, help="训练轮数")
    parser.add_argument("--batch_size", type=int, default=64, help="批次大小")
    parser.add_argument("--learning_rate", type=float, default=2e-4, help="学习率")
    parser.add_argument("--device", type=str, default="cuda:0" if torch.cuda.is_available() else "cpu", help="训练设备")
    parser.add_argument("--dtype", type=str, default="bfloat16", help="数据类型")
    
    # 实验跟踪和数据加载参数
    parser.add_argument("--use_swanlab", action="store_true", help="是否使用SwanLab进行实验跟踪")
    parser.add_argument("--num_workers", type=int, default=8, help="数据加载的工作进程数")
    parser.add_argument("--data_path", type=str, default="./seq_monkey_datawhale.jsonl", help="训练数据路径")
    
    # 训练优化参数
    parser.add_argument("--accumulation_steps", type=int, default=8, help="梯度累积步数")
    parser.add_argument("--grad_clip", type=float, default=1.0, help="梯度裁剪阈值")
    parser.add_argument("--warmup_iters", type=int, default=0, help="学习率预热迭代次数")
    
    # 日志和保存参数
    parser.add_argument("--log_interval", type=int, default=100, help="日志记录间隔")
    parser.add_argument("--save_interval", type=int, default=1000, help="模型保存间隔")
    
    # 多GPU训练参数
    parser.add_argument("--gpus", type=str, default='0,1,2,3,4,5,6,7', help="使用的GPU ID，用逗号分隔 (例如: '0,1,2')")

    args = parser.parse_args()

    # ==================== GPU环境设置 ====================
    # 设置可见的GPU设备
    if args.gpus is not None:
        os.environ["CUDA_VISIBLE_DEVICES"] = args.gpus
        # 自动设置主设备为第一个可用GPU
        if torch.cuda.is_available():
            args.device = "cuda:0"
        else:
            args.device = "cpu"

    # ==================== 实验跟踪初始化 ====================
    if args.use_swanlab:
        # 注意：使用前需要先登录 swanlab.login(api_key='your key')
        run = swanlab.init(
            project="Happy-LLM",  # 项目名称
            experiment_name="Pretrain-215M",  # 实验名称
            config=args,  # 保存所有超参数
        )

    # ==================== 模型配置 ====================
    # 定义语言模型的配置参数
    lm_config = ModelConfig(
        dim=1024,      # 模型维度
        n_layers=18,   # Transformer层数
    )

    # ==================== 训练环境设置 ====================
    max_seq_len = lm_config.max_seq_len  # 最大序列长度
    args.save_dir = os.path.join(args.out_dir)  # 模型保存目录
    
    # 创建必要的目录
    os.makedirs(args.out_dir, exist_ok=True)
    
    # 设置随机种子以确保结果可复现
    torch.manual_seed(42)
    
    # 确定设备类型（用于选择合适的上下文管理器）
    device_type = "cuda" if "cuda" in args.device else "cpu"

    # 设置混合精度训练的上下文管理器
    # CPU训练时使用nullcontext，GPU训练时使用autocast
    ctx = nullcontext() if device_type == "cpu" else torch.cuda.amp.autocast()

    # ==================== 模型和数据初始化 ====================
    # 初始化模型和分词器
    model, tokenizer = init_model()
    
    # 创建训练数据集
    train_ds = PretrainDataset(args.data_path, tokenizer, max_length=max_seq_len)
    
    # 创建数据加载器
    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,  # 批次大小
        pin_memory=True,             # 将数据加载到固定内存中，加速GPU传输
        drop_last=False,             # 不丢弃最后一个不完整的批次
        shuffle=True,                # 随机打乱数据
        num_workers=args.num_workers # 数据加载的并行工作进程数
    )

    # ==================== 优化器和训练组件初始化 ====================
    # 初始化混合精度训练的梯度缩放器
    # 只有在使用float16或bfloat16时才启用
    scaler = torch.cuda.amp.GradScaler(enabled=(args.dtype in ['float16', 'bfloat16']))
    
    # 初始化Adam优化器
    optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)

    # ==================== 开始训练 ====================
    # 计算每个epoch的迭代次数
    iter_per_epoch = len(train_loader)
    
    # 开始训练循环
    for epoch in range(args.epochs):
        train_epoch(epoch)

#### `ddp_pretrain.py` 详解：Tiny-LLM 预训练脚本全流程解读
```
这个脚本把前面所有组件（`Transformer` 模型、`PretrainDataset`）串起来，完成一次完整的**预训练循环**。核心是四件事：**学习率调度（warmup + 余弦退火）、混合精度训练、梯度累积、梯度裁剪**，外加多卡 DataParallel、周期保存检查点和 SwanLab 实验跟踪。
```
---
##### 一、整体结构
```
argparse 解析超参 → GPU/日志环境设置 → init_model() 创建模型
→ PretrainDataset + DataLoader → Adam 优化器 + GradScaler
→ for epoch: train_epoch()   ← 训练主循环
```

```
顶部 `RUN_TRAINING = False` 是安全开关——改 True 且准备好数据/GPU 才会执行，防止误运行。
```

---
##### 二、`get_lr`：学习率调度器（三段式）
```python
def get_lr(it, all):
```
这是标准的 **warmup + cosine decay** 调度：
```
lr
 │        ╭─────╮╲
 │       ╱       ╲╲
 │      ╱          ╲╲        ← 余弦缓慢下降
 │     ╱             ╲___
 │    ╱  warmup        min_lr（保持不变）
 └───┴─────────────────────┴──→ step
     0   ↑              ↑
   warmup_iters      总步数
```
| 阶段 | 代码 | 说明 |
|---|---|---|
| **Warmup（线性升）** | `it < warmup_iters` → `lr * it / warmup_iters` | 从 0 线性升到目标 lr。训练初期模型参数随机、梯度大且方向不可靠，直接用大学习率会震荡甚至发散 |
| **余弦退火** | `0.5 * (1 + cos(π * decay_ratio))` | decay_ratio 从 0→1，系数从 1→0，lr 平滑降到 min_lr。先慢后快再慢的衰减比线性/阶梯衰减更稳定 |
| **收尾** | `it > all` → `min_lr` | 超出总步数后保持最小学习率（初始 lr 的 1/10） |
注意调度是按**全局步数** `epoch * iter_per_epoch + step` 计算的，跨 epoch 连续。
另外脚本里手动更新 lr（`param_group['lr'] = lr`）而非用 `lr_scheduler`，效果等价。
---
##### 三、`train_epoch`：训练主循环逐段解读
###### 1. 数据搬运
```python
for step, (X, Y, loss_mask) in enumerate(train_loader):
    X, Y, loss_mask = X.to(args.device), Y.to(...), loss_mask.to(...)
```
```
取出 PretrainDataset 产出的三元组并搬到 GPU。
```
###### 2. 前向 + 混合精度上下文
```python
with ctx:
    out = model(X, Y)
    loss = out.last_loss / args.accumulation_steps
```
- `ctx`：GPU 上是 `torch.cuda.amp.autocast()`，自动把矩阵乘等运算降到 bf16/fp16 计算、省显存提速；CPU 上是 `nullcontext` 什么都不做
- **`loss / accumulation_steps` 是梯度累积的关键**：等效于把 8 个 mini-batch 的梯度平均后当作一个大 batch。因为每个小 batch 的 loss 都除以 8，累加 8 次的梯度才等于大 batch 一次算出的梯度
###### 3. loss_mask 加权
```python
loss_mask = loss_mask.view(-1)
loss = torch.sum(loss * loss_mask) / loss_mask.sum()
```

```
模型内部返回的是**逐 token 的交叉熵**（`reduction='none'`）。这里：
- 乘 mask → padding 位置的损失置 0
- 除以 `mask.sum()` → 只对**真实 token 数**求平均，而不是整个 batch 长度
这样不同样本长度不同时，每个真实 token 的权重一致。
```
###### 4. 反向传播与优化器更新
```python
scaler.scale(loss).backward()
if (step + 1) % args.accumulation_steps == 0:
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
```
逐步解释：
| 步骤 | 作用 |
|---|---|
| `scaler.scale(loss).backward()` | 对放大后的 loss 反传。fp16 下小梯度可能下溢为 0，GradScaler 先把 loss 乘一个大系数，让梯度处于可表示范围 |
| `scaler.unscale_` | 在裁剪前把梯度除回缩放系数，恢复真实梯度值 |
| `clip_grad_norm_(..., 1.0)` | **梯度裁剪**：全模型梯度的 L2 范数超过 1.0 就等比缩小，防止个别 batch 出现梯度爆炸破坏训练 |
| `scaler.step(optimizer)` | 内部先检查是否出现 inf/nan（fp16 溢出检测），正常才执行参数更新 |
| `scaler.update()` | 根据是否溢出动态调整下次的缩放系数 |
| `zero_grad(set_to_none=True)` | 累积 8 步后才清零梯度；`set_to_none=True` 把梯度直接置 None 而非清 0，**省一次显存写入** |

```
注意节奏：**每一步都 backward 累积梯度，每 8 步才 update 一次参数**——等效 batch_size = 64 × 8 = 512。
```
###### 5. 日志记录
```python
if step % args.log_interval == 0:
    Logger(... loss.item() * args.accumulation_steps ...)
```
- `loss * accumulation_steps` 把除回去的 loss **还原为真实值**再打印（否则显示的是 1/8 的 loss）
- epoch 剩余时间估算：`spend_time / (step+1) * iter_per_epoch - spend_time`，按平均每步耗时线性外推
###### 6. 检查点保存
```python
if (step + 1) % args.save_interval == 0:
    model.eval()
    ckp = f'{save_dir}/pretrain_{dim}_{n_layers}_{vocab_size}.pth'
    state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
    torch.save(state_dict, ckp)
    model.train()
```
- 文件名编码了模型结构信息，便于识别不同配置的 checkpoint
- **`model.module.state_dict()`**：DataParallel 包装后模型多了一层 `.module`，直接保存会把 `module.` 前缀写进所有 key，加载时对不上。这里剥掉包装只存纯净权重
- `model.eval()` / `model.train()`：保存时切换（虽对 state_dict 非必需，但保持习惯一致）；每 20000 步额外存一个带步数标记的版本，防止后续覆盖导致无法回溯
---
##### 四、`init_model`：模型初始化
```python
tokenizer = AutoTokenizer.from_pretrained('./tokenizer_k/')
model = Transformer(lm_config)
```
- 用 `AutoTokenizer` 加载之前训练/保存的 tokenizer 配置
- 按配置实例化 Transformer（LLaMA2 风格：RMSNorm + GQA + RoPE + SwiGLU）
多 GPU 处理：
```python
if num_gpus > 1:
    model = torch.nn.DataParallel(model)
model = model.to(args.device)
```

```
`DataParallel` 是最简单的数据并行：每个 batch 被切分到多卡各算一份前向，梯度汇总到主卡。（生产更常用 DDP，这里用 DP 属于简化实现，注释里的 `ddp_pretrain` 文件名暗示原版本有 DDP 变体。）
`count_parameters` 统计 `requires_grad=True` 的参数总量，打印如 "215.3 百万"（对应 215M 配置：dim=1024, n_layers=18）。
```
---
##### 五、主程序：超参与环境
###### 关键超参
| 参数 | 默认值 | 含义 |
|---|---|---|
| `batch_size` | 64 | 每 GPU 步样本数 |
| `learning_rate` | 2e-4 | 峰值学习率 |
| `dtype` | bfloat16 | 混合精度类型 |
| `accumulation_steps` | 8 | 梯度累积 → 等效 batch 512 |
| `grad_clip` | 1.0 | 梯度范数上限 |
| `warmup_iters` | 0 | 预热步数（默认关闭） |
| `lm_config` | dim=1024, n_layers=18 | 215M 规模的小模型 |
###### 环境设置要点
```python
os.environ["CUDA_VISIBLE_DEVICES"] = args.gpus     # 指定用哪几张卡
torch.manual_seed(42)                              # 固定种子保证可复现
ctx = nullcontext() if cpu else torch.cuda.amp.autocast()  # 混合精度上下文
```
###### DataLoader 细节
```python
DataLoader(train_ds, batch_size=64, pin_memory=True, shuffle=True,
           drop_last=False, num_workers=8)
```
- `pin_memory=True`：数据先放入锁页内存，GPU 传输（`.to(device)`）更快
- `num_workers=8`：8 个子进程并行做 seek/读文件/tokenize，避免数据加载成为训练瓶颈
- `shuffle=True`：每轮打乱样本顺序
###### 组件初始化
```python
scaler = torch.cuda.amp.GradScaler(enabled=(args.dtype in ['float16', 'bfloat16']))
optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)
```
- GradScaler 在 bf16/fp16 时启用（bf16 动态范围大其实不太需要 scaler，这里保守开启）
- Adam 优化器，配合上面手动调度 lr
###### 训练启动
```python
iter_per_epoch = len(train_loader)
for epoch in range(args.epochs):
    train_epoch(epoch)
```
---
##### 六、一次训练 step 的完整时间线
```
取 batch (X, Y, mask) → 搬到 GPU
→ 算 lr (warmup/cosine) → 手动写入 optimizer
→ autocast 前向 → 逐 token loss ÷ 8 → mask 加权平均
→ scaler.scale().backward()   （梯度累积中，不动参数）
├─ 未满 8 步 → 继续下一个 batch
└─ 满 8 步 → unscale → 裁剪范数≤1.0 → scaler.step 更新参数
                        → scaler.update → zero_grad(set_to_none)
→ 每 100 步打日志 / 每 1000 步存 checkpoint
```
##### 七、设计要点总结
| 技术 | 解决的问题 |
|---|---|
| Warmup | 训练初期梯度不可靠，防止大 lr 发散 |
| 余弦退火 | 后期平滑降 lr，收敛更稳 |
| 混合精度 autocast + GradScaler | 显存减半、速度翻倍，同时防 fp16 下溢 |
| 梯度累积（÷8） | 小显存下模拟大 batch（64→512） |
| 梯度裁剪 | 防梯度爆炸 |
| loss_mask 加权 | padding 不参与损失，长短样本权重公平 |
| `.module` 剥离保存 | 保证 checkpoint 可干净加载 |
| 偏移量索引 Dataset + 多 worker | 大语料流式加载不卡训练 |

```
一句话：这是一个**面向单机/单机多卡的教学级预训练脚本**——把 LLaMA2 风格的小模型（215M）、内存友好的数据管线、warmup+余弦调度、bf16 混合精度、梯度累积与裁剪组装成一个标准训练循环，每 1000 步落盘一次，可用 SwanLab 观察训练曲线。
```

##### 深入理解梯度累积与累积步数除法
###### 一、问题的根源：PyTorch 的梯度是"累加"的
PyTorch 中调用 `loss.backward()` 时，梯度**不是覆盖，而是累加**到每个参数的 `.grad` 上：
```python
x = torch.zeros(1, requires_grad=True)
loss1 = x * 1
loss1.backward()
print(x.grad)  # tensor([1.])
loss2 = x * 1
loss2.backward()
print(x.grad)  # tensor([2.])  ← 累加了！
```
这个设计本身是特性（方便多次 backward 累加），但意味着**我们手动控制"何时清零"**——通常每个训练循环开头 `optimizer.zero_grad()`。梯度累积正是利用了这个累加机制。
###### 二、梯度累积的完整流程
```python
optimizer.zero_grad()                          # ① 清零梯度
for i, (input, target) in enumerate(dataloader):
    out = model(input, target)
    loss = out.last_loss / accumulation_steps  # ② 关键：除以累积步数
    loss.backward()                            # ③ 梯度累加到 .grad
    if (i + 1) % accumulation_steps == 0:      # ④ 累积满 N 步
        optimizer.step()                       #    用累加后的平均梯度更新参数
        optimizer.zero_grad()                  #    清零，开始下一轮累积
```
###### 三、为什么必须除以累积步数？——数学推导
不除的情况（错误做法）
假设真实需求是 batch_size = 32 的训练，显存只够 batch_size = 8，累积步数 N = 4。
单个小 batch 的梯度为 gᵢ，`backward()` 累加 4 次后：
$$g_{\text{accum}} = g_1 + g_2 + g_3 + g_4$$
而标准大 batch（32 个样本一次 forward）的损失是：
$$L_{\text{big}} = \frac{1}{32}\sum_{j=1}^{32} \ell_j \quad \Rightarrow \quad g_{\text{big}} = \frac{1}{32}\sum_{j=1}^{32} \nabla\ell_j$$
注意标准损失的 **mean** 会把梯度"除回去"。而不除以 N 的累积梯度没有这个平均化：
$$g_{\text{accum}} \approx 4 \cdot g_{\text{big}}$$
即**梯度尺度放大了 N 倍**，等效学习率变成原来的 N 倍。后果：
- 权重更新步长过大，损失震荡、发散；
- 对学习率敏感的组件（如 warmup、weight decay 的相对强度）全部失真。
除以 N 的正确做法
$$\frac{1}{N}\sum_{i=1}^{N} g_i = \frac{1}{N}\sum_{i=1}^{N}\left(\frac{1}{8}\sum_{j=1}^{8}\nabla\ell_j\right) = \frac{1}{32}\sum_{j=1}^{32}\nabla\ell_j = g_{\text{big}}$$
**只要每个 micro-batch 内的损失是 mean 归约的，除以累积步数后，累积梯度在数学上严格等价于大 batch 的梯度。**
⚠️ 一个常见坑：如果每个 micro-batch 样本数不同（最后一个 batch 不足），用"样本数加权"更精确：`loss = sum_per_token_loss / total_tokens`，再对全累积周期的 token 数求平均。
###### 四、为什么累积期间不能用 BN 的 BatchNorm 类操作？
需注意两点等价性差异：
1. **BatchNorm 统计量**：BN 的 running statistics 每个前向都更新，micro-batch 的均值/方差估计比真大 batch 更噪。LLM 用的是 **RMSNorm/LayerNorm**（逐 token 归一化，与 batch 维无关），所以累积完全无损——这也是 Transformer 时代梯度累积近乎"免费"的原因之一。
2. **Dropout 随机性**：每个 micro-batch 独立采样 dropout mask，与一次大 batch forward 的结果略有差异，通常影响可忽略。
###### 五、几个容易踩的坑
 坑 1：只在 `optimizer.step()` 后清零，累积中途误清零
```python
# 错误：每个 batch 都 zero_grad
for batch in dataloader:
    optimizer.zero_grad()   # ❌ 前面的累积全被抹掉
    ...
```
 坑 2：学习率调度器 step 的时机
LR scheduler 一般应跟着 `optimizer.step()` 走，即每 N 个 micro-batch 才 scheduler.step() 一次，否则调度器"走得"比实际更新快 N 倍。
 坑 3：Gradient Scaling（混合精度训练）
用 `torch.cuda.amp` 时，scaler 的 scale 也要除以累积步数处理：
```python
with autocast():
    loss = out.last_loss / accumulation_steps
scaler.scale(loss).backward()   # scale 乘在除过的 loss 上，逻辑一致
```
注意 scale 因子是动态的，若用 `loss.backward()` 前手动 detach 累加 logits 等跨步张量要小心 scale 变化。
####### 坑 4：`model.zero_grad(set_to_none=True)` 更高效
`set_to_none=True` 把 `.grad` 置为 None 而不是置零张量，省一次显存写操作，PyTorch 2.x 默认推荐。
####### 坑 5：累积期间的 forward 结果不要跨步保存
每步的 loss 应该只用于 backward，别把 `loss_total += loss` 用于日志后再除——若 loss 已经除过 N，日志里要乘回 N 才是真实损失值，否则日志显示的 loss 偏小。
###### 六、与"除以累积步数"对应的记忆要点
| 记忆点 | 说明 |
|---|---|
| 本质 | 让 N 次 backward 累加的梯度 = 一次大 batch 的平均梯度 |
| 关键操作 | 每次 backward 前 `loss /= N`，每 N 步 `optimizer.step() + zero_grad()` |
| 等价条件 | 每个 micro-batch 内 loss 是 mean 归约 |
| 典型收益 | 显存不变的情况下，等效 batch size = micro_bs × N |
| 潜在代价 | 优化器真实更新频率降低 N 倍，warmup/调度需按实际更新次数配置 |
**一句话总结**：`loss / accumulation_steps` 不是损失函数的"正则化"，而是对 PyTorch 梯度累加机制的**尺度补偿**——没有这一步，梯度累积就退化成了"变相放大学习率 N 倍"，训练必然不稳定。

#### 梯度缩放与梯度裁剪详解
```
这两行代码是混合精度训练（AMP）收尾阶段的标准流程。它们解决的是两个不同的问题，但协作使用。
```
---
##### 一、梯度缩放：混合精度训练的"保命机制"
###### 1. 背景：为什么需要混合精度？
训练大模型用 **FP16/BF16 半精度**计算（速度快、显存省一半），但有个致命问题——**数值范围小**。
FP16 的数值范围约 $6 \times 10^{-8} \sim 6 \times 10^{4}$。反向传播中，**很多梯度非常小**（如 $10^{-7}$），在 FP16 中直接**下溢为 0**：
```
真实梯度: 1e-7  →  FP16 表示: 0  ❌（精度丢失）
大量梯度变成 0，模型就"学不动了"。
```

###### 2. 解决方案：放大损失，放大梯度
```
既然小数会下溢，那就**先乘一个大数（如 65536），让梯度"够得着"可表示范围**，更新参数前再除回来：
```
$$\text{scaled\_loss} = L \times S \quad\Rightarrow\quad \nabla_{\text{scaled}} = \nabla L \times S$$
```python
scaler = torch.cuda.amp.GradScaler()
with torch.autocast(device_type="cuda"):       # forward 用 FP16
    loss = model(x, y).last_loss
scaler.scale(loss).backward()                  # 反向时梯度被放大 S 倍
scaler.unscale_(optimizer)                     # ← 除回 S 倍，还原真实梯度
optimizer.step()                               # 用真实梯度更新参数
```
###### 3. `scaler.unscale_(optimizer)` 在做什么？
```
把 optimizer 引用的所有参数的 `.grad` **除以缩放因子 S**，还原成真实梯度。之后 optimizer.step() 才能进行正确的参数更新。
关键点：**它只是"还原"，不更新参数**——这正是它和 `scaler.step(optimizer)` 的区别。
```
###### 4. 为什么必须先 unscale 才能裁剪？
```
这就引出了第二部分。
```

---
##### 二、梯度裁剪：防止梯度爆炸
###### 1. 问题：梯度爆炸
训练 LLM（尤其是带 RoPE、深层堆叠、half-precision）时，某些 step 梯度可能突然变得巨大（数值不稳定、数据异常、学习率过高等）。参数一次性更新过大，损失直接 NaN / 发散。
###### 2. 原理：范数裁剪
`clip_grad_norm_` 计算所有参数梯度的**全局 L2 范数**：
$$\|g\|_2 = \sqrt{\sum_p \|g_p\|_2^2}$$
如果范数超过阈值（LLM 常用 **1.0**），就整体等比例缩小：
$$g \leftarrow g \times \frac{\text{max\_norm}}{\|g\|_2}$$
```python
torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)  # 阈值常取1.0
```
要点：
- **只缩小、不放大**：范数小于阈值时梯度原样保留，不影响正常训练；
- **全局裁剪**：不是每个参数单独裁，而是按所有梯度组成的总向量缩放，保持梯度方向不变，只调整步长；
- 裁剪发生在 **`backward()` 之后、`optimizer.step()` 之前**——因为它是基于"已经算好的梯度"做修正。
---
##### 三、为什么两步的顺序至关重要？
这就是把两段代码放在一起的原因：
```python
scaler.unscale_(optimizer)    # ① 先还原真实梯度
torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)  # ② 再裁剪
scaler.step(optimizer)        # ③ 更新参数（内部还会处理 inf/nan 检查）
```
**如果顺序反了会发生什么？**
假设真实梯度范数是 5，缩放因子 S=65536：
| 顺序 | 计算的范数 | 结果 |
|---|---|---|
| ✅ 先 unscale 再 clip | 5.0 | 裁剪到 1.0，正确 |
| ❌ 先 clip 再 unscale | 5.0 × 65536 ≈ 32 万 | 裁剪后除以 65536，**梯度被压到 1.5e-5**，几乎不更新 |
在放大的梯度上直接裁剪，阈值会被"虚假的巨大范数"触发，裁剪力度远超预期，最终的有效梯度被过度压缩，模型训练严重失真甚至停滞。
---
##### 四、完整视角：一个 AMP + 裁剪 + 梯度累积的训练 step
把之前讨论的梯度累积也串起来：
```python
# scaler = torch.cuda.amp.GradScaler()
optimizer.zero_grad()
for i, batch in enumerate(dataloader):
    with torch.autocast("cuda"):                    # ① FP16 前向
        out = model(batch.input_ids, batch.labels)
        loss = out.last_loss / accumulation_steps   # ② 梯度累积的尺度补偿
    scaler.scale(loss).backward()                   # ③ 放大 S 倍的反向传播（梯度累加）
    if (i + 1) % accumulation_steps == 0:
        scaler.unscale_(optimizer)                  # ④ 还原真实梯度
        torch.nn.utils.clip_grad_norm_(             # ⑤ 对真实梯度裁剪
            model.parameters(), args.grad_clip
        )
        scaler.step(optimizer)                      # ⑥ 更新（含 inf/nan 检查，动态调 S）
        scaler.update()                             # ⑦ 动态调整缩放因子
        optimizer.zero_grad()
```

```
补充：`scaler.step()` 内部会检查梯度中是否有 `inf/NaN`（溢出标志）——如果有，**跳过本次更新**；`scaler.update()` 则动态调整 S：经常溢出就调小 S，长期稳定就逐步调大 S。这就是"动态损失缩放"（Dynamic Loss Scaling）。
```

---
##### 五、总结对比
| 技术 | 解决的问题 | 作用时机 | 修改了什么 |
|---|---|---|---|
| **梯度缩放** (scale/unscale) | FP16 小梯度下溢 | backward 前放大，step 前还原 | 梯度的**数值尺度**（临时性的） |
| **梯度裁剪** (clip_grad_norm_) | 梯度爆炸、训练不稳定 | backward 后、step 前 | 超阈值时的**梯度模长**（永久修改） |
| **梯度累积** (÷N) | 显存不够大 batch | 每次 backward 前 | 梯度的**累积尺度**（补偿累加效应） |
三者的共同点：都在**梯度空间**上做文章，都不改变模型结构和损失定义，目的都是让参数更新这一步"又稳又对"。
**一句话记忆**：缩放是为了"算得出"（数值表示），裁剪是为了"更新稳"（步长可控），而 `unscale_` 是两者衔接的桥梁——必须在真实梯度上做裁剪，顺序不能错。

### 5.3.5 SFT 训练

SFT 与预训练的主要差异：

- 数据换成对话格式，`loss_mask` 只覆盖 assistant 回复；
- 从预训练权重初始化，`strict=False` 并去掉 `_orig_mod.` 前缀（兼容 `torch.compile` 保存的权重）；
- 学习率更小、训练轮数更少，避免破坏预训练能力。

In [31]:
RUN_TRAINING = False  # 改为 True 并准备好数据/GPU 后才会执行 main

import os
import platform
import argparse
import time
import warnings
import math
import pandas as pd
import torch
from torch import optim
from torch.utils.data import DataLoader
from contextlib import nullcontext

from transformers import AutoTokenizer


try:
    import swanlab
except ImportError:
    swanlab = None  # 未安装时只影响日志记录

# 忽略警告
warnings.filterwarnings('ignore')


def Logger(content):
    """日志记录器"""
    print(content)

def get_lr(it, all):
    """获取学习率"""
    # 1) linear warmup for warmup_iters steps
    # 1) 预热迭代的线性预热
    warmup_iters = args.warmup_iters
    lr_decay_iters = all
    min_lr = args.learning_rate / 10

    if it < warmup_iters:
        return args.learning_rate * it / warmup_iters
    
    # 2) if it > lr_decay_iters, return min learning rate
    # 2) 如果迭代次数超过学习率衰减迭代次数，则返回最小学习率
    if it > lr_decay_iters:
        return min_lr
    
    # 3) in between, use cosine decay down to min learning rate
    # 3) 在两者之间，使用余弦衰减至最小学习率
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (args.learning_rate - min_lr)

def train_epoch(epoch):
    """训练一个epoch"""
    start_time = time.time()
    for step, (X, Y, loss_mask) in enumerate(train_loader):
        X = X.to(args.device)
        Y = Y.to(args.device)
        loss_mask = loss_mask.to(args.device)

        # 获取学习率并更新优化器
        lr = get_lr(epoch * iter_per_epoch + step, args.epochs * iter_per_epoch)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # 前向传播
        with ctx:
            out = model(X, Y)
            loss = out.last_loss / args.accumulation_steps
            loss_mask = loss_mask.view(-1)
            loss = torch.sum(loss * loss_mask) / loss_mask.sum()

        # 反向传播
        scaler.scale(loss).backward()

        # 更新权重
        if (step + 1) % args.accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        # 打印日志
        if step % args.log_interval == 0:
            spend_time = time.time() - start_time
            Logger(
                'Epoch:[{}/{}]({}/{}) loss:{:.3f} lr:{:.7f} epoch_Time:{}min:'.format(
                    epoch + 1,
                    args.epochs,
                    step,
                    iter_per_epoch,
                    loss.item() * args.accumulation_steps,
                    optimizer.param_groups[-1]['lr'],
                    spend_time / (step + 1) * iter_per_epoch // 60 - spend_time // 60))
            if args.use_swanlab:
                swanlab.log({
                    "loss": loss.item() * args.accumulation_steps,
                    "lr": optimizer.param_groups[-1]['lr']
                })

        # 保存模型
        if (step + 1) % args.save_interval == 0:
            model.eval()
            ckp = f'{args.save_dir}/sft_dim{lm_config.dim}_layers{lm_config.n_layers}_vocab_size{lm_config.vocab_size}.pth'

            # 处理多卡保存
            state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
            torch.save(state_dict, ckp)
            model.train()
        
        # 定期保存模型
        if (step + 1) % 20000 == 0:
            model.eval()
            ckp = f'{args.save_dir}/sft_dim{lm_config.dim}_layers{lm_config.n_layers}_vocab_size{lm_config.vocab_size}_step{step+1}.pth'

            state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
            torch.save(state_dict, ckp)
            model.train()


def init_model():
    """初始化模型"""
    def count_parameters(model):
        """计算模型参数量"""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained('./tokenizer_k/')

    # 初始化模型
    model = Transformer(lm_config)

    # 加载预训练权重
    ckp = './base_model_215M/pretrain_1024_18_6144.pth'
    state_dict = torch.load(ckp, map_location=args.device)
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict, strict=False)
    
    # 多卡初始化
    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        Logger(f"Using {num_gpus} GPUs with DataParallel!")
        model = torch.nn.DataParallel(model)
    
    model = model.to(args.device)
    Logger(f'LLM总参数量：{count_parameters(model) / 1e6:.3f} 百万')
    return model, tokenizer


if __name__ == "__main__" and RUN_TRAINING:
    parser = argparse.ArgumentParser(description="Tiny-LLM Pretraining")
    parser.add_argument("--out_dir", type=str, default="sft_model_215M", help="输出目录")
    parser.add_argument("--epochs", type=int, default=1, help="训练轮数")
    parser.add_argument("--batch_size", type=int, default=64, help="批处理大小")
    parser.add_argument("--learning_rate", type=float, default=2e-4, help="学习率")
    parser.add_argument("--device", type=str, default="cuda:0" if torch.cuda.is_available() else "cpu", help="使用的设备")
    parser.add_argument("--dtype", type=str, default="bfloat16", help="数据类型")
    parser.add_argument("--use_swanlab", action="store_true", help="是否使用SwanLab进行实验跟踪")
    parser.add_argument("--num_workers", type=int, default=8, help="数据加载的工作进程数")
    parser.add_argument("--data_path", type=str, default="./BelleGroup_sft.jsonl", help="训练数据路径")
    parser.add_argument("--accumulation_steps", type=int, default=8, help="梯度累积步数")
    parser.add_argument("--grad_clip", type=float, default=1.0, help="梯度裁剪阈值")
    parser.add_argument("--warmup_iters", type=int, default=0, help="预热迭代次数")
    parser.add_argument("--log_interval", type=int, default=100, help="日志记录间隔")
    parser.add_argument("--save_interval", type=int, default=1000, help="模型保存间隔")
    # 添加多卡参数
    parser.add_argument("--gpus", type=str, default='0,1,2,3,4,5,6,7', help="逗号分隔的GPU ID (例如 '0,1,2')")

    args = parser.parse_args()

    # 设置可见GPU
    if args.gpus is not None:
        os.environ["CUDA_VISIBLE_DEVICES"] = args.gpus
        # 自动设置主设备为第一个GPU
        if torch.cuda.is_available():
            args.device = "cuda:0"
        else:
            args.device = "cpu"

    # 初始化swanlab
    if args.use_swanlab:
        run = swanlab.init(
            project="Happy-LLM",
            experiment_name="SFT-215M",
            config=args,
        )

    # 模型配置
    lm_config = ModelConfig(
        dim=1024,
        n_layers=18,
    )
    max_seq_len = lm_config.max_seq_len
    args.save_dir = os.path.join(args.out_dir)
    os.makedirs(args.out_dir, exist_ok=True)
    torch.manual_seed(42)
    device_type = "cuda" if "cuda" in args.device else "cpu"

    # 上下文管理器
    ctx = nullcontext() if device_type == "cpu" else torch.cuda.amp.autocast()

    # 初始化模型和分词器
    model, tokenizer = init_model()
    
    # 创建数据集和数据加载器
    train_ds = SFTDataset(args.data_path, tokenizer, max_length=max_seq_len)
    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,
        pin_memory=True,
        drop_last=False,
        shuffle=True,
        num_workers=args.num_workers
    )

    # 缩放器和优化器
    scaler = torch.cuda.amp.GradScaler(enabled=(args.dtype in ['float16', 'bfloat16']))
    optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate)

    # 开始训练
    iter_per_epoch = len(train_loader)
    for epoch in range(args.epochs):
        train_epoch(epoch)

#### SFT 训练脚本详解：从预训练模型到对话模型的微调

```
这份脚本是**监督微调（SFT）**阶段，整体结构与之前的预训练脚本几乎一致——复用了同一套 `get_lr` 调度、梯度累积、混合精度、梯度裁剪、检查点保存框架。真正的区别集中在**四处**：加载预训练权重、换用 `SFTDataset`、优化器换成 AdamW、输出/checkpoint 命名。下面按“相同点简述 + 差异点详解”来解读。
```
---
##### 一、整体流程对比
```
预训练脚本                        SFT 脚本
─────────────────────           ─────────────────────
从零初始化 Transformer           从零初始化 + 加载预训练权重  ← 差异①
PretrainDataset（纯文本）        SFTDataset（多轮对话+mask）← 差异②
optim.Adam                      optim.AdamW                ← 差异③
保存 pretrain_*.pth             保存 sft_*.pth             ← 差异④
数据: seq_monkey 语料            数据: BelleGroup 对话数据
实验名: Pretrain-215M            实验名: SFT-215M
```

```
其余部分（warmup+余弦调度、autocast、GradScaler、梯度累积 ÷8、clip_grad_norm、DataParallel、SwanLab）与预训练脚本完全相同，不再赘述。
```

---
##### 二、差异①：`init_model` 中加载预训练权重（最关键的新增逻辑）
```python
model = Transformer(lm_config)
# 加载预训练权重
ckp = './base_model_215M/pretrain_1024_18_6144.pth'
state_dict = torch.load(ckp, map_location=args.device)
```
###### 清理 `_orig_mod.` 前缀
```python
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
```

```
这段处理的是一个常见的坑：如果预训练时模型曾经过 `torch.compile(model)` 编译，保存的 state_dict 所有 key 会带上 `_orig_mod.` 前缀（如 `_orig_mod.layers.0.attention.wq.weight`）。这里遍历所有 key，把前缀剥掉，保证能和当前裸模型的结构对上。
```

###### 非严格加载
```python
model.load_state_dict(state_dict, strict=False)
```
```
- `strict=False`：预训练 checkpoint 与当前模型的**参数 key 不必完全一致**——缺失的 key 保持随机初始化，多余的 key 被忽略，**不报错**
- 为什么需要？本项目的 tokenizer 词表从预训练到 SFT 可能发生变化（如 6144 vs 其他 vocab_size），导致 embedding / output 层形状对不上；`strict=False` 让这类差异静默跳过（代价是可能遗漏权重，需自己确认）
这一步就是 SFT 的本质起点：**在预训练学到的语言能力之上继续训练**，而不是从零开始。
```
---
##### 三、差异②：换用 `SFTDataset`
```python
train_ds = SFTDataset(args.data_path, tokenizer, max_length=max_seq_len)
```
```
数据文件也从通用语料换成对话数据：
```

```python
parser.add_argument("--data_path", default="./BelleGroup_sft.jsonl")
```
回顾 SFTDataset 的行为差异：
| 环节 | 预训练 | SFT |
|---|---|---|
| 数据内容 | `{"text": "..."}` 纯文本 | user/assistant 多轮对话 messages |
| 文本构造 | 手动拼 `<s>` | `apply_chat_template` 渲染 `<\|im_start\|>`... `<\|im_end\|>` 模板 |
| loss_mask | 仅屏蔽 padding | **只在 assistant 回复（含 eos）上为 1** |
| 学到的东西 | 语言/知识本身 | “给定对话如何回答”的行为 |
训练循环里 `loss = torch.sum(loss * loss_mask) / loss_mask.sum()` 这一行因此有了新的意义：预训练时它屏蔽 padding；SFT 时它**屏蔽用户问题和模板，只让模型学回答**——这是 SFT 不让模型学会“自己提问”的关键机制。
---
##### 四、差异③：Adam → AdamW
```python
# 预训练脚本
optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)
# SFT 脚本
optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate)
```

```
区别在**权重衰减（weight decay）**的实现方式：
- **Adam**：L2 正则被加进梯度，会经过自适应学习率的缩放，导致不同参数实际衰减力度不一致，正则效果失真
- **AdamW**：解耦权重衰减——衰减项直接作用在参数上（`p -= lr * wd * p`），不经过梯度通道，这才是真正意义上的 weight decay
现代 LLM 训练（GPT/LLaMA 等）标配 AdamW。SFT 换用它更规范，且轻微的正则有助于防止小数据集微调时**过拟合/灾难性遗忘**。
```

---
##### 五、差异④：checkpoint 命名与实验标识
```python
ckp = f'{args.save_dir}/sft_dim{...}_layers{...}_vocab_size{...}.pth'
# 预训练是 pretrain_...
run = swanlab.init(project="Happy-LLM", experiment_name="SFT-215M", ...)
```

```
输出目录 `sft_model_215M`、文件名前缀 `sft_`、实验名 `SFT-215M`，纯粹是产物标识，与预训练产物区分开，方便后续推理脚本加载正确的权重。
```

---
##### 六、其余与预训练完全相同的部分（简述）
| 组件 | 内容 |
|---|---|
| `get_lr` | warmup 线性升 → 余弦退火到 min_lr（初始 lr 的 1/10）→ 保持 |
| `ctx` | GPU 上 `autocast`（bf16 混合精度），CPU 上 `nullcontext` |
| `GradScaler` | bf16/fp16 时启用，防小梯度下溢 |
| 梯度累积 | loss ÷ 8 累积 8 步再 update，等效 batch 512 |
| `clip_grad_norm_(1.0)` | unscale 后裁剪梯度范数，防爆炸 |
| 日志 | 每 100 步打印 loss（×8 还原真实值）/lr/剩余时间；SwanLab 记录 |
| 保存 | 每 1000 步存常规 ckp、每 20000 步存带步数版本；DataParallel 时剥 `.module` |
| 多卡 | `CUDA_VISIBLE_DEVICES` + `DataParallel` |
---
##### 七、SFT 一次 step 的时间线
```
读对话 JSON → apply_chat_template 渲染 → tokenize/padding
→ 构造 (X, Y, loss_mask)（mask 只标 assistant 回复+<|im_end|>）
→ 搬 GPU → 更新 lr → autocast 前向（预训练权重已加载）
→ 逐 token CE × mask 加权平均 → ÷8 反传累积
→ 每 8 步：unscale → 裁剪 → AdamW step（解耦权重衰减）
→ 周期性保存 sft_*.pth
```
##### 八、总结
```
这份 SFT 脚本 = 预训练脚本的训练框架 + 三个阶段性改动：
1. **`init_model` 加载 `pretrain_1024_18_6144.pth`**（含 `_orig_mod.` 前缀清理 + `strict=False` 非严格加载）——站在预训练的肩膀上
2. **`SFTDataset` + BelleGroup 对话数据**——ChatML 模板 + assistant-only loss mask，只学“回答”
3. **AdamW**——解耦权重衰减，防微调过拟合
加上产物命名 `sft_*` 区分。它体现了 LLM 训练的标准两段式范式：**预训练学语言，SFT 学行为**，前者赋予模型知识与生成能力，后者通过精心设计的损失掩码把这种能力引导为“遵循指令、进行对话”的可用形态。
```

### 5.3.6 使用模型生成文本

`TextGenerator` 负责：

1. 加载检查点与 Tokenizer；
2. 根据设备选择 `float32` / `bfloat16` / `float16`；
3. 应用聊天模板构造 prompt；
4. 调用 `Transformer.generate` 采样，并只解码新生成的 token。

原文示例输出（SFT 模型）：

```text
Question: 你好呀
AI answer: 你好!有什么我可以帮你的吗?

Question: 中国的首都是哪里？
AI answer: 中国的首都是北京。

Question: 1+1等于多少？
AI answer: 1+1等于2。
```

> `RUN_SAMPLING` 默认为 `False`；需要本地已有检查点和 Tokenizer 时再改为 `True`。

In [32]:
RUN_SAMPLING = False  # 改为 True 并准备好检查点/Tokenizer 后才会执行 main

import os
import pickle
from contextlib import nullcontext
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import argparse

class TextGenerator:
    def __init__(self, 
                 checkpoint='./base_model_215M/pretrain_1024_18_6144.pth',  # 模型检查点路径
                 tokenizer_model_path='./tokenizer_k/',  # 分词器模型路径
                 seed=42,  # 随机种子，确保可重复性
                 device=None,  # 设备，优先使用 CUDA，如果没有可用的 CUDA，则使用 CPU
                 dtype="bfloat16"):  # 数据类型，默认为 float32，可以选择 float16 或 bfloat16
        """
        初始化 TextGenerator 类，加载模型、设置设备和分词器等。
        """
        # 模型加载配置
        self.checkpoint = checkpoint  # 保存的模型检查点路径
        self.tokenizer_model_path = tokenizer_model_path  # 分词器模型文件路径
        self.seed = seed  # 随机数种子，用于生成的可重复性
        self.device = device or ('cuda:0' if torch.cuda.is_available() else 'cpu')  # 根据硬件条件选择设备
        self.dtype = dtype  # 模型的浮点数类型
        self.device_type = 'cuda' if 'cuda' in self.device else 'cpu'  # 判断当前设备是否为 CUDA
        
        # 设置随机种子，确保生成的可重复性
        torch.manual_seed(seed)  # 设置 CPU 随机种子
        torch.cuda.manual_seed(seed)  # 设置 CUDA 随机种子
        torch.backends.cuda.matmul.allow_tf32 = True  # 允许 CUDA 使用 TF32 精度进行矩阵乘法运算
        torch.backends.cudnn.allow_tf32 = True  # 允许 cuDNN 使用 TF32 精度加速
        
        # 根据 dtype 选择适当的自动混合精度上下文
        ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[self.dtype]
        self.ctx = nullcontext() if self.device_type == 'cpu' else torch.amp.autocast(device_type=self.device_type, dtype=ptdtype)
        
        # 加载模型检查点文件
        checkpoint_dict = torch.load(self.checkpoint, map_location=self.device)  # 加载模型参数 # 初始化模型参数
        self.model = Transformer(ModelConfig(dim=1024, n_layers=18))  # 实例化 Transformer 模型
        sunwanted_prefix = '_orig_mod.'
        for k, v in list(checkpoint_dict.items()):
            if k.startswith(sunwanted_prefix):
                checkpoint_dict[k[len(sunwanted_prefix):]] = checkpoint_dict.pop(k)
        self.model.load_state_dict(checkpoint_dict, strict=False)
        
        # 计算模型参数量
        num_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"Model has {num_params / 1e6:.3f} M parameters.")
        # 设置模型为评估模式（evaluation mode），防止训练模式下的 dropout 等操作影响结果
        self.model.eval()
        # 将模型放置到正确的设备上（GPU 或 CPU）
        self.model.to(self.device)
        # 初始化分词器
        self.tokenizer = AutoTokenizer.from_pretrained(self.tokenizer_model_path)  # 根据指定的路径加载分词器

    def chat_template(self, prompt):
        message = [
            {"role": "system", "content": "你是一个AI助手，你的名字叫小明。"},
            {"role": "user", "content": prompt}
        ]
        return self.tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

    def sft_sample(self, 
               start="Hello!",  # 生成文本的起始提示词，可以是任意字符串
               num_samples=3,  # 生成样本的数量，默认生成 3 个样本
               max_new_tokens=256,  # 每个样本生成的最大 token 数，默认最多生成 256 个 token
               temperature=0.7,  # 控制生成的随机性，1.0 为标准，值越大越随机
               top_k=300):  # 保留概率最高的 top_k 个 token，限制生成时的选择范围
        """
        根据给定的起始文本生成样本。
        
        :param start: 生成文本的起始提示词
        :param num_samples: 要生成的文本样本数
        :param max_new_tokens: 每个样本生成的最大 token 数
        :param temperature: 控制生成的随机性，值越小生成越确定，值越大生成越随机
        :param top_k: 限制生成时选择的 token 范围
        :return: 生成的文本样本列表
        """
        start = self.chat_template(start)
        # 将起始文本编码为 token id 序列
        start_ids = self.tokenizer(start).data['input_ids']
        # print('start_ids:', start_ids)
        x = (torch.tensor(start_ids, dtype=torch.long, device=self.device)[None, ...])  # 将编码后的 token id 转为 PyTorch 张量
        generated_texts = []  # 用于保存生成的文本样本
        with torch.no_grad():  # 禁用梯度计算，提升效率
            with self.ctx:  # 进入自动混合精度的上下文（如果是 GPU 并使用 float16 时）
                for k in range(num_samples):  # 循环生成指定数量的样本
                    y = self.model.generate(x, self.tokenizer.eos_token_id, max_new_tokens, temperature=temperature, top_k=top_k)  # 生成文本
                    generated_texts.append(self.tokenizer.decode(y[0].tolist()))  # 解码生成的 token 序列为可读文本
        return generated_texts  # 返回生成的文本样本


    def pretrain_sample(self, 
               start="Hello!",  # 生成文本的起始提示词，可以是任意字符串
               num_samples=3,  # 生成样本的数量，默认生成 3 个样本
               max_new_tokens=256,  # 每个样本生成的最大 token 数，默认最多生成 256 个 token
               temperature=0.7,  # 控制生成的随机性，1.0 为标准，值越大越随机
               top_k=300):  # 保留概率最高的 top_k 个 token，限制生成时的选择范围
        """
        根据给定的起始文本生成样本。
        
        :param start: 生成文本的起始提示词
        :param num_samples: 要生成的文本样本数
        :param max_new_tokens: 每个样本生成的最大 token 数
        :param temperature: 控制生成的随机性，值越小生成越确定，值越大生成越随机
        :param top_k: 限制生成时选择的 token 范围
        :return: 生成的文本样本列表
        """
        # 如果 start 是以 'FILE:' 开头，表示从文件中读取起始文本
        if start.startswith('FILE:'):
            with open(start[5:], 'r', encoding='utf-8') as f:
                start = f.read()  # 读取文件内容作为起始文本
        
        # 将起始文本编码为 token id 序列
        start_ids = self.tokenizer(start).data['input_ids']
        # print('start_ids:', start_ids)
        x = (torch.tensor(start_ids, dtype=torch.long, device=self.device)[None, ...])  # 将编码后的 token id 转为 PyTorch 张量
        # print(x.shape)
        generated_texts = []  # 用于保存生成的文本样本
        with torch.no_grad():  # 禁用梯度计算，提升效率
            with self.ctx:  # 进入自动混合精度的上下文（如果是 GPU 并使用 float16 时）
                for k in range(num_samples):  # 循环生成指定数量的样本
                    y = self.model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)  # 生成文本
                    generated_texts.append(self.tokenizer.decode(y[0].tolist()))  # 解码生成的 token 序列为可读文本
        
        return generated_texts  # 返回生成的文本样本
    
if __name__ == "__main__" and RUN_SAMPLING:
    print("------------------- Pretrain Sample ------------------- \n")

    pretrain_prompt_datas = [
        '<|im_start|>北京大学是',
        '<|im_start|>中国矿业大学（北京）地球科学与测绘工程学院',
    ]

    generator = TextGenerator(checkpoint='./base_model_215M/pretrain_1024_18_6144.pth')  # 初始化生成器
    for i in range(len(pretrain_prompt_datas)):
        samples = generator.pretrain_sample(start=pretrain_prompt_datas[i], num_samples=1, max_new_tokens=120, temperature=0.75)
        print(f"\nSample {i+1}:\n{pretrain_prompt_datas[i]}{samples[0]}\n{'-'*20}")  # 打印生成的样本并用分隔线分割

    print("\n ------------------- SFT Sample ------------------- \n")

    sft_prompt_datas = [
        '你好呀',
        "中国的首都是哪里？",
        "1+12等于多少？",
        "你是谁？"
    ]
    generator = TextGenerator(checkpoint='./sft_model_215M/sft_dim1024_layers18_vocab_size6144.pth')  # 初始化生成器
    for i in range(len(sft_prompt_datas)):
        samples = generator.sft_sample(start=sft_prompt_datas[i], num_samples=1, max_new_tokens=128, temperature=0.6)
        print(f"\nSample {i+1}:\nQuestion: {sft_prompt_datas[i]} \nAI answer: {samples[0]}\n{'-'*20}")  # 打印生成的样本并用分隔线分割

#### 采样脚本详解：加载训练好的模型进行文本生成验证

```
这份脚本是整个训练流程的**最后一环**：分别加载**预训练模型**和 **SFT 模型**两个 checkpoint，用同一套生成逻辑各自采样，直观对比“预训练模型只会续写、SFT 模型会对话”的能力差异。核心是 `TextGenerator` 类和两种采样方法。
```

---
##### 一、整体结构
```
TextGenerator（封装：加载模型 + 分词器 + 设备/精度配置）
    ├── pretrain_sample()   ← 用预训练权重做纯文本续写
    ├── sft_sample()        ← 用 SFT 权重做对话生成
    └── chat_template()     ← 为 sft_sample 构造 ChatML 提示
main:
    Part 1: 加载 pretrain_*.pth → 续写 "北京大学是..." 等
    Part 2: 加载 sft_*.pth       → 回答 "你是谁？" 等

顶部 `RUN_SAMPLING = False` 安全开关，与训练脚本风格一致——改 True 且准备好 checkpoint 后才执行。
```

---
##### 二、`__init__`：模型加载与环境配置
###### 1. 可复现性设置
```python
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
```

```
固定随机种子，保证采样结果可重复。生成中的随机性来自 `torch.multinomial` 采样，同一种子下结果完全一致。
```

###### 2. TF32 加速
```python
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
```

```
允许 Ampere+ GPU 在矩阵乘法中使用 TF32（19 位精度格式）：比 fp32 快、精度损失可忽略，纯推理加速开关。
```

###### 3. 混合精度上下文
```python
self.ctx = nullcontext() if cpu else torch.amp.autocast(device_type, dtype=ptdtype)
```

```
与训练脚本相同：GPU 上用 bf16 autocast 加速推理，CPU 上什么都不做。
```

###### 4. 加载 checkpoint（与 SFT 脚本同款逻辑）
```python
self.model = Transformer(ModelConfig(dim=1024, n_layers=18))
checkpoint_dict = torch.load(self.checkpoint, map_location=self.device)
# 剥掉 '_orig_mod.' 前缀（处理 torch.compile 保存的权重）
self.model.load_state_dict(checkpoint_dict, strict=False)
```

```
关键区别：**模型结构代码在本地**（`Transformer(...)` 由本文件 import），不需要 HuggingFace 的 `AutoModelForCausalLM`——虽然 import 了它，但实际只用了 `AutoTokenizer`。
```

###### 5. 推理模式的必要设置
```python
self.model.eval()       # 关闭 dropout 等训练行为
self.model.to(self.device)
```
- `eval()` 很重要：模型里有 Dropout，若忘切 eval，生成结果会随机性地变差
- `map_location` 保证 checkpoint 无论原来存于哪张卡，都能加载到当前设备
---
##### 三、`chat_template`：把用户输入包装成对话格式
```python
def chat_template(self, prompt):
    message = [
        {"role": "system", "content": "你是一个AI助手，你的名字叫小明。"},
        {"role": "user", "content": prompt}
    ]
    return self.tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
```

```
三个要点：
1. **system prompt**：注入人设（“你叫小明”），SFT 数据若含 system 角色对话，模型会学会遵循
2. **`apply_chat_template`**：渲染成训练时同款格式：

<|im_start|>system
你是一个AI助手，你的名字叫小明。<|im_end|>
<|im_start|>user
你好<|im_end|>
```

```
3. **`add_generation_prompt=True`**：这是与训练时（False）的**关键差异**——末尾追加 `<|im_start|>assistant\n`，明确告诉模型“现在轮到你回答了”。这正是 SFTDataset 中锚点匹配所对应的推理期用法
**训练/推理格式一致**是 SFT 生效的前提：如果推理时模板和训练时不匹配，模型见到的输入分布是陌生的，效果会大打折扣。
```

---
##### 四、`pretrain_sample`：预训练模型续写
```python
if start.startswith('FILE:'):
    with open(start[5:], 'r', encoding='utf-8') as f:
        start = f.read()        # 支持从文件读长 prompt
start_ids = self.tokenizer(start).data['input_ids']
x = torch.tensor(start_ids, dtype=torch.long, device=self.device)[None, ...]
```

```
- `[None, ...]` 把一维 id 序列升维成 `(1, T)` 的 batch
- 之后循环 `num_samples` 次，每次调用模型的 `generate()`（之前解读过的自回归采样：上下文截断 → temperature 缩放 → top-k 过滤 → multinomial 采样 → 碰 eos 停止），解码回文本
**注意它没有传 stop_id**：预训练模型只是续写文本，不需要对话停止符。
```

---
##### 五、`sft_sample`：SFT 模型对话（与 pretrain_sample 的两处差异）
```python
start = self.chat_template(start)     # 差异①：先套 ChatML 模板
y = self.model.generate(x, self.tokenizer.eos_token_id, ...)
                                      # 差异②：传入 stop_id
```
| 差异点 | pretrain_sample | sft_sample |
|---|---|---|
| 输入构造 | 原始文本直接编码 | system+user 包装成 ChatML，末尾带 assistant 提示 |
| 停止符 | 无（写到 max_new_tokens） | **`eos_token_id`（`<\|im_end\|>`）** |

```
停止符正是 SFT 时刻意训练的：SFTDataset 的 loss mask 包含了回复末尾的 `<|im_end|>`，所以模型学会“答完就输出 im_end"→ 推理时 `generate` 捕获它提前停止，不会无限续写。
其他细节相同：`torch.no_grad()` 禁梯度（省显存提速）、`with self.ctx` 进 autocast、逐样本 decode。
```

---
##### 六、`main`：双阶段验证实验
###### Part 1 — 预训练模型
```python
pretrain_prompt_datas = [
    '<|im_start|>北京大学是',
    '<|im_start|>中国矿业大学（北京）地球科学与测绘工程学院',
]
generator = TextGenerator(checkpoint='./base_model_215M/pretrain_1024_18_6144.pth')
samples = generator.pretrain_sample(start=..., num_samples=1, max_new_tokens=120, temperature=0.75)
```
- prompt 手动带 `<|im_start|>`：预训练语料可能含这种格式，模型对它有续写习惯
- 典型输出行为：**续写百科式文本**（如“北京大学是……一所位于中国北京市的综合性大学……”），可能中途跳到下一段“文档”，因为预训练模型只学了语言分布、不会停下来
###### Part 2 — SFT 模型
```python
sft_prompt_datas = ['你好呀', '中国的首都是哪里？', '1+12等于多少？', '你是谁？']
generator = TextGenerator(checkpoint='./sft_model_215M/sft_dim1024_layers18_vocab_size6144.pth')
samples = generator.sft_sample(start=..., num_samples=1, max_new_tokens=128, temperature=0.6)
```
- 同一个 `TextGenerator` 类，只是**换了 checkpoint**——架构相同（dim=1024, 18 层, vocab 6144），权重不同
- temperature 从 0.75 降到 0.6：对话任务要更确定性，减少胡言乱语
- 期望输出行为：以 `<|im_end|>` 结尾的**成段回答**，而非无限续写
###### 这组对照实验的意义
| | 预训练模型 | SFT 模型 |
|---|---|---|
| 输入 | 裸文本 | ChatML 对话模板 |
| 输出 | 文本续写（不会停） | 回答 + `<\|im_end\|>` 停止 |
| 能力 | 语言/知识 | 指令遵循、对话 |
同一个 215M 架构，仅因训练数据和损失掩码不同，行为就完全不同——这正是 **"预训练学语言，SFT 学行为”** 最直观的实证。
---
##### 七、设计要点总结
| 组件/技巧 | 作用 |
|---|---|
| 固定 seed | 采样结果可复现 |
| TF32 + bf16 autocast | 推理提速，精度损失可忽略 |
| 剥 `_orig_mod.` 前缀 + `strict=False` | 兼容 torch.compile 产物与词表差异 |
| `model.eval()` | 关闭 dropout，保证推理确定性 |
| `chat_template` + `add_generation_prompt=True` | 推理格式与训练格式严格对齐 |
| 传入 `eos_token_id` 作 stop_id | 利用 SFT 训练的结束行为，答完即停 |
| `torch.no_grad()` | 不建计算图，省显存提速 |
| 同类两 checkpoint 对照 | 直观展示预训练 vs SFT 的能力差异 |
一句话：这是一个**轻量的推理验证脚本**——用封装好的 `TextGenerator`（模型加载、精度、模板、采样参数全部就位），先让预训练权重做纯文本续写，再让 SFT 权重在 ChatML 模板下回答问题并以 `<|im_end|>` 停止，直观检验两个训练阶段的成果，完成了“预训练 → SFT → 推理评估”的完整闭环。

## 预训练和 SFT 的关联与区别
### 一、基本概念
**预训练（Pre-training）**：在海量无标注（或自监督）语料上，用“预测下一个 token”这类目标训练语言模型的通用语言能力。例如第五章预训练小 LLM 时，就是用大段文本，对每个位置做下一词预测并计算交叉熵损失。
**SFT（Supervised Fine-Tuning，有监督微调）**：在预训练模型基础上，用人工构造的高质量“指令—回答”样本继续训练，让模型学会以对话/指令形式与人类交互，即 Chat 模型的核心阶段。
### 二、区别
| 维度 | 预训练 | SFT |
|------|--------|-----|
| **训练目标** | 下一 token 预测（自监督），对齐语言本身 | 同样是下一 token 预测，但只对“回答部分”计算损失，指令部分通常被 mask 掉 |
| **数据** | 海量通用语料（万亿级 token），无需人工标注 | 少量高质量指令数据（几万~几十万条），需人工构造/标注 |
| **数据格式** | 连续长文本，常按 max_seq_len 切分拼接 | 结构化的多轮对话格式，含角色标记、特殊 token（如对话模板） |
| **起点权重** | 随机初始化（或 continuation 预训练） | 加载预训练 checkpoint 作为初始权重 |
| **训练成本** | 极高（大量 GPU、数周~数月） | 较低（少量 GPU、数小时~数天） |
| **学到的东西** | 语言知识、世界知识、推理能力的“底子” | 遵循指令、对话格式、人类偏好风格 |
| **评估效果** | 困惑度（PPL）、下游任务零样本能力 | 指令遵循能力、人工评测 |
### 三、关联
1. **同一模型架构、同一训练目标**：两者都用同一个 Transformer（如课程中实现的 LLaMA2）和因果语言建模损失，只是数据和训练流程不同。
2. **SFT 是预训练的延续**：SFT 通常在预训练权重上以较小学习率继续训练，相当于在通用能力之上“对齐”到任务格式，属于同一训练 pipeline 的不同阶段。
3. **配合关系**：预训练决定模型能力上限（知识、推理），SFT 决定模型能否把能力以指令/对话方式表达出来——只做预训练的 base 模型不擅长回答问题，只做 SFT 无法凭空产生预训练没有的知识。
4. **完整流程**：预训练 → SFT → （可选）RLHF/DPO 对齐。Happy-LLM 课程中第五章“预训练一个小型 LLM”和第六章的训练实践正是按这一流程组织的。

## 附：Notebook 与源码文件对应关系

| Notebook 部分 | 对应源码 |
| --- | --- |
| 5.1 模型实现 | `code/k_model.py` |
| 5.2 Tokenizer | `code/train_tokenizer.py` |
| 5.3.1 数据处理 | 原文数据准备脚本 |
| 5.3.3 Dataset | `code/dataset.py` |
| 5.3.4 预训练 | `code/ddp_pretrain.py` |
| 5.3.5 SFT | `code/ddp_sft_full.py` |
| 5.3.6 文本生成 | `code/model_sample.py` |

补充说明：

- 本 Notebook 的模型代码以原文 5.1 节的精简版为准，便于逐段阅读和运行。
- `code/k_model.py` 额外支持 `attention_mask`、左 padding 等推理细节，并包含更完整的生成逻辑。
- 训练脚本中的 `RUN_TRAINING` / `RUN_SAMPLING` 是本 Notebook 新增的保护开关；在原始 `.py` 脚本中对应 `if __name__ == "__main__":`。